# Cell 1 — Load QC-Qualified Genome Manifest

In [ ]:
import pandas as pd

manifest_path = "/content/genome_manifest_qc.csv"

df_manifest_qc = pd.read_csv(
    manifest_path,
    dtype={
        "Genome ID": str,
        "Genome ID String": str,
        "Taxon ID": str
    }
)

print("QC-qualified genome manifest loaded successfully.")
print(f"Rows            : {len(df_manifest_qc):,}")
print(f"Columns         : {df_manifest_qc.shape[1]}")
print(f"Unique Genome IDs: {df_manifest_qc['Genome ID String'].nunique():,}")
print()

print("Phenotype distribution:")
print(df_manifest_qc["Resistant Phenotype"].value_counts())

display(df_manifest_qc.head())

QC-qualified genome manifest loaded successfully.
Rows            : 4,233
Columns         : 6
Unique Genome IDs: 4,233

Phenotype distribution:
Resistant Phenotype
Susceptible    2630
Resistant      1603
Name: count, dtype: int64


,Genome ID,Genome ID String,Genome Name,Taxon ID,Antibiotic,Resistant Phenotype
0,1284787.3,1284787.3,Klebsiella pneumoniae UHKPC40,1284787,meropenem,Resistant
1,1284788.3,1284788.3,Klebsiella pneumoniae UHKPC23,1284788,meropenem,Resistant
2,1284789.4,1284789.4,Klebsiella pneumoniae UHKPC28,1284789,meropenem,Resistant
3,1284790.3,1284790.3,Klebsiella pneumoniae UHKPC05,1284790,meropenem,Resistant
4,1284791.4,1284791.4,Klebsiella pneumoniae UHKPC47,1284791,meropenem,Resistant


# Cell 2 — Pilot Genome FASTA Retrieval Check

In [ ]:
import os
import re
import time
import requests
import pandas as pd


# ---------------------------------------------------------
# 1. Select balanced pilot cohort
# ---------------------------------------------------------

pilot_r = (
    df_manifest_qc[df_manifest_qc["Resistant Phenotype"] == "Resistant"]
    .head(5)
)

pilot_s = (
    df_manifest_qc[df_manifest_qc["Resistant Phenotype"] == "Susceptible"]
    .head(5)
)

df_pilot = (
    pd.concat([pilot_r, pilot_s], ignore_index=True)
    [["Genome ID String", "Genome Name", "Resistant Phenotype"]]
)

print("Pilot cohort:")
display(df_pilot)


# ---------------------------------------------------------
# 2. Load expected genome QC metadata
# ---------------------------------------------------------

qc_metadata_path = "/content/genome_qc_metadata.csv"

df_qc_metadata = pd.read_csv(
    qc_metadata_path,
    dtype={
        "genome_id": str
    }
)

# Identify genome ID column robustly
if "Genome ID String" in df_qc_metadata.columns:
    qc_id_col = "Genome ID String"
elif "genome_id" in df_qc_metadata.columns:
    qc_id_col = "genome_id"
elif "Genome ID" in df_qc_metadata.columns:
    qc_id_col = "Genome ID"
else:
    raise ValueError("Genome ID column not found in genome_qc_metadata.csv")

df_qc_metadata[qc_id_col] = df_qc_metadata[qc_id_col].astype(str)

expected_qc = (
    df_qc_metadata[
        df_qc_metadata[qc_id_col].isin(df_pilot["Genome ID String"])
    ]
    .copy()
)

print("\nExpected QC metadata:")
display(
    expected_qc[
        [qc_id_col, "genome_length", "contigs"]
    ]
)


# ---------------------------------------------------------
# 3. Prepare output directory
# ---------------------------------------------------------

pilot_dir = "/content/genomes/pilot_fasta"
os.makedirs(pilot_dir, exist_ok=True)


# ---------------------------------------------------------
# 4. FASTA audit helper
# ---------------------------------------------------------

def audit_fasta(fasta_text):

    lines = [
        line.strip()
        for line in fasta_text.splitlines()
        if line.strip()
    ]

    headers = [
        line for line in lines
        if line.startswith(">")
    ]

    sequence_lines = [
        line for line in lines
        if not line.startswith(">")
    ]

    sequence = "".join(sequence_lines).upper()

    valid_sequence = (
        len(sequence) > 0
        and re.fullmatch(r"[ACGTNRYKMSWBDHV\-]+", sequence) is not None
    )

    return {
        "fasta_records": len(headers),
        "total_sequence_bp": len(sequence),
        "valid_sequence": valid_sequence
    }


# ---------------------------------------------------------
# 5. Retrieve COMPLETE pilot genome assemblies
# ---------------------------------------------------------

results = []

for _, row in df_pilot.iterrows():

    genome_id = row["Genome ID String"]

    # Explicitly request enough sequence records to avoid
    # the API's result-window truncation.
    url = (
        "https://www.bv-brc.org/api/genome_sequence/"
        f"?eq(genome_id,{genome_id})"
        "&limit(10000)"
        "&http_accept=application/dna%2Bfasta"
    )

    try:

        response = requests.get(
            url,
            timeout=120,
            headers={"Accept": "application/dna+fasta"}
        )

        response.raise_for_status()

        fasta_text = response.text
        audit = audit_fasta(fasta_text)

        qc_row = expected_qc[
            expected_qc[qc_id_col] == genome_id
        ]

        if len(qc_row) != 1:
            raise ValueError(
                f"Expected QC metadata not uniquely found for {genome_id}"
            )

        expected_length = int(qc_row.iloc[0]["genome_length"])
        expected_contigs = int(qc_row.iloc[0]["contigs"])

        length_difference = (
            audit["total_sequence_bp"] - expected_length
        )

        relative_length_difference = (
            abs(length_difference) / expected_length
        )

        # Allow very small metadata/sequence differences
        length_match = relative_length_difference <= 0.001

        record_match = (
            audit["fasta_records"] == expected_contigs
        )

        complete_fasta = (
            audit["valid_sequence"]
            and length_match
        )

        output_path = os.path.join(
            pilot_dir,
            f"{genome_id}.fna"
        )

        if complete_fasta:
            with open(output_path, "w") as f:
                f.write(fasta_text)

        results.append({
            "Genome ID": genome_id,
            "Phenotype": row["Resistant Phenotype"],
            "HTTP Status": response.status_code,

            "Expected Contigs": expected_contigs,
            "FASTA Records": audit["fasta_records"],
            "Record Count Match": record_match,

            "Expected Length (bp)": expected_length,
            "FASTA Length (bp)": audit["total_sequence_bp"],
            "Length Difference (bp)": length_difference,
            "Length Match": length_match,

            "Valid Sequence": audit["valid_sequence"],
            "Complete FASTA": complete_fasta,
            "Saved": complete_fasta
        })

    except Exception as e:

        results.append({
            "Genome ID": genome_id,
            "Phenotype": row["Resistant Phenotype"],
            "HTTP Status": None,
            "Complete FASTA": False,
            "Saved": False,
            "Error": str(e)
        })

    time.sleep(0.2)


# ---------------------------------------------------------
# 6. Pilot audit summary
# ---------------------------------------------------------

df_pilot_fasta_audit = pd.DataFrame(results)

print("\nComplete FASTA retrieval audit:")
display(df_pilot_fasta_audit)

print("\nSummary")
print("-" * 60)

successful = (
    df_pilot_fasta_audit["Complete FASTA"]
    .fillna(False)
    .sum()
)

print(
    f"Complete FASTA retrievals : "
    f"{successful}/{len(df_pilot_fasta_audit)}"
)

print(
    f"Saved FASTA files         : "
    f"{df_pilot_fasta_audit['Saved'].fillna(False).sum()}"
    f"/{len(df_pilot_fasta_audit)}"
)

if successful == len(df_pilot_fasta_audit):
    print(
        "\nPASS: All pilot genomes were retrieved as complete "
        "FASTA assemblies."
    )
else:
    print(
        "\nWARNING: At least one genome did not match "
        "the expected assembly length."
    )

Pilot cohort:


,Genome ID String,Genome Name,Resistant Phenotype
0,1284787.3,Klebsiella pneumoniae UHKPC40,Resistant
1,1284788.3,Klebsiella pneumoniae UHKPC23,Resistant
2,1284789.4,Klebsiella pneumoniae UHKPC28,Resistant
3,1284790.3,Klebsiella pneumoniae UHKPC05,Resistant
4,1284791.4,Klebsiella pneumoniae UHKPC47,Resistant
5,1284805.4,Klebsiella pneumoniae UHKPC17,Susceptible
6,1284814.3,Klebsiella pneumoniae UHKPC 52,Susceptible
7,1284818.4,Klebsiella pneumoniae UHKPC179,Susceptible
8,1284834.3,Klebsiella pneumoniae KP-11,Susceptible
9,1328384.3,Klebsiella pneumoniae BWH 28,Susceptible



Expected QC metadata:


,genome_id,genome_length,contigs
9,1284787.3,5490156,155
10,1284788.3,5503448,195
11,1284789.4,5398963,137
12,1284790.3,5502006,187
13,1284791.4,5498288,195
27,1284805.4,5581309,185
36,1284814.3,5424339,154
40,1284818.4,5515994,174
56,1284834.3,5718845,301
90,1328384.3,5533872,23



Complete FASTA retrieval audit:


,Genome ID,Phenotype,HTTP Status,Expected Contigs,FASTA Records,Record Count Match,Expected Length (bp),FASTA Length (bp),Length Difference (bp),Length Match,Valid Sequence,Complete FASTA,Saved
0,1284787.3,Resistant,200,155,155,True,5490156,5490156,0,True,True,True,True
1,1284788.3,Resistant,200,195,195,True,5503448,5503448,0,True,True,True,True
2,1284789.4,Resistant,200,137,137,True,5398963,5398963,0,True,True,True,True
3,1284790.3,Resistant,200,187,187,True,5502006,5502006,0,True,True,True,True
4,1284791.4,Resistant,200,195,195,True,5498288,5498288,0,True,True,True,True
5,1284805.4,Susceptible,200,185,185,True,5581309,5581309,0,True,True,True,True
6,1284814.3,Susceptible,200,154,154,True,5424339,5424339,0,True,True,True,True
7,1284818.4,Susceptible,200,174,174,True,5515994,5515994,0,True,True,True,True
8,1284834.3,Susceptible,200,301,301,True,5718845,5718845,0,True,True,True,True
9,1328384.3,Susceptible,200,23,23,True,5533872,5533872,0,True,True,True,True



Summary
------------------------------------------------------------
Complete FASTA retrievals : 10/10
Saved FASTA files         : 10/10

PASS: All pilot genomes were retrieved as complete FASTA assemblies.


# Cell 3 — Parallel Full Genome FASTA Retrieval

In [ ]:
from google.colab import drive

import os
import subprocess


# =========================================================
# 1. Configuration
# =========================================================

DRIVE_MOUNT_POINT = "/content/drive"


# =========================================================
# 2. Check whether Google Drive is already mounted
# =========================================================

def is_drive_mounted():

    try:

        result = subprocess.run(
            [
                "mountpoint",
                "-q",
                DRIVE_MOUNT_POINT
            ],
            timeout=5
        )

        return (
            result.returncode == 0
        )

    except Exception:

        return False


# =========================================================
# 3. Mount Google Drive only when necessary
# =========================================================

if is_drive_mounted():

    print(
        "Google Drive is already mounted."
    )

else:

    print(
        "Mounting Google Drive..."
    )

    drive.mount(
        DRIVE_MOUNT_POINT,
        force_remount=False,
        timeout_ms=120000
    )

    print(
        "Google Drive mounted successfully."
    )


# =========================================================
# 4. Persistent project paths
# =========================================================

DRIVE_ROOT = (
    "/content/drive/MyDrive/AMR-Genome-ML"
)


DRIVE_FASTA_DIR = os.path.join(
    DRIVE_ROOT,
    "data/genomes/fasta_gz"
)


DRIVE_CHECKPOINT_DIR = os.path.join(
    DRIVE_ROOT,
    "checkpoints"
)


DRIVE_AUDIT_PATH = os.path.join(
    DRIVE_CHECKPOINT_DIR,
    "genome_fasta_retrieval_audit.csv"
)


# =========================================================
# 5. Local compatibility paths
# =========================================================

LOCAL_FASTA_DIR = (
    "/content/genomes/fasta_gz"
)


LOCAL_AUDIT_PATH = (
    "/content/genome_fasta_retrieval_audit.csv"
)


QC_METADATA_PATH = (
    "/content/genome_qc_metadata.csv"
)


MAX_WORKERS = 8
BACKUP_EVERY = 200


# =========================================================
# 6. Lightweight Drive accessibility check
#
# Use subprocess + timeout so an unhealthy DriveFS
# cannot freeze the notebook indefinitely.
# =========================================================

def path_exists_with_timeout(
    path,
    path_type="file",
    timeout=15
):

    test_flag = (
        "-d"
        if path_type == "dir"
        else "-f"
    )

    try:

        result = subprocess.run(
            [
                "test",
                test_flag,
                path
            ],
            timeout=timeout
        )

        return (
            result.returncode == 0
        )

    except subprocess.TimeoutExpired:

        raise RuntimeError(
            "Google Drive is mounted but DriveFS "
            "is not responding. Restart the Colab "
            "session and mount Drive again."
        )


fasta_dir_ok = (
    path_exists_with_timeout(
        DRIVE_FASTA_DIR,
        path_type="dir"
    )
)


audit_file_ok = (
    path_exists_with_timeout(
        DRIVE_AUDIT_PATH,
        path_type="file"
    )
)


if not fasta_dir_ok:

    raise FileNotFoundError(
        f"FASTA directory not found: "
        f"{DRIVE_FASTA_DIR}"
    )


if not audit_file_ok:

    raise FileNotFoundError(
        f"Audit file not found: "
        f"{DRIVE_AUDIT_PATH}"
    )


# =========================================================
# 7. Status
# =========================================================

print()
print("Persistent storage ready")
print("-" * 60)

print(
    f"FASTA directory : "
    f"{DRIVE_FASTA_DIR}"
)

print(
    f"Audit file      : "
    f"{DRIVE_AUDIT_PATH}"
)

print()
print(
    "PASS: Google Drive storage is accessible."
)

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.

Persistent storage ready
------------------------------------------------------------
FASTA directory : /content/drive/MyDrive/AMR-Genome-ML/data/genomes/fasta_gz
Audit file      : /content/drive/MyDrive/AMR-Genome-ML/checkpoints/genome_fasta_retrieval_audit.csv

PASS: Google Drive storage is accessible.


# Cell 4 — Prepare Full Retrieval Cohort and QC Reference

In [ ]:
import pandas as pd


# =========================================================
# Prepare manifest IDs
# =========================================================

df_manifest_qc["Genome ID String"] = (
    df_manifest_qc["Genome ID String"]
    .astype(str)
    .str.strip()
)


assert len(df_manifest_qc) == 4233

assert (
    df_manifest_qc[
        "Genome ID String"
    ].nunique()
    == 4233
)


# =========================================================
# Load QC metadata
#
# IMPORTANT:
# genome_id MUST be read as string to preserve trailing zeros.
# =========================================================

df_qc_metadata = pd.read_csv(
    QC_METADATA_PATH,
    dtype={
        "genome_id": str
    }
)


if "Genome ID String" in df_qc_metadata.columns:

    qc_id_col = "Genome ID String"

elif "genome_id" in df_qc_metadata.columns:

    qc_id_col = "genome_id"

elif "Genome ID" in df_qc_metadata.columns:

    qc_id_col = "Genome ID"

else:

    raise ValueError(
        "Genome ID column not found."
    )


df_qc_metadata[qc_id_col] = (
    df_qc_metadata[qc_id_col]
    .astype(str)
    .str.strip()
)


# =========================================================
# Required QC fields
# =========================================================

required_columns = {
    qc_id_col,
    "genome_length",
    "contigs"
}

missing_columns = (
    required_columns
    - set(df_qc_metadata.columns)
)

if missing_columns:

    raise ValueError(
        f"Missing QC columns: {missing_columns}"
    )


# =========================================================
# Build lookup
# =========================================================

qc_reference = (
    df_qc_metadata[
        df_qc_metadata[
            qc_id_col
        ].isin(
            df_manifest_qc[
                "Genome ID String"
            ]
        )
    ]
    [
        [
            qc_id_col,
            "genome_length",
            "contigs"
        ]
    ]
    .drop_duplicates(
        subset=[qc_id_col]
    )
    .copy()
)


qc_lookup = (
    qc_reference
    .set_index(qc_id_col)
)


manifest_ids = set(
    df_manifest_qc[
        "Genome ID String"
    ]
)

qc_ids = set(
    qc_lookup.index
)


missing_ids = (
    manifest_ids - qc_ids
)

extra_ids = (
    qc_ids - manifest_ids
)


assert len(missing_ids) == 0

assert len(extra_ids) == 0


print("Cohort integrity")
print("-" * 60)

print(
    f"Manifest genomes      : "
    f"{len(manifest_ids):,}"
)

print(
    f"QC reference genomes  : "
    f"{len(qc_ids):,}"
)

print(
    f"Missing QC references : "
    f"{len(missing_ids):,}"
)

print(
    f"Unexpected QC IDs     : "
    f"{len(extra_ids):,}"
)

Cohort integrity
------------------------------------------------------------
Manifest genomes      : 4,233
QC reference genomes  : 4,233
Missing QC references : 0
Unexpected QC IDs     : 0


# Cell 5 — Define FASTA Retrieval and Validation Functions

In [ ]:
import os
import re
import gzip
import shutil
import subprocess


# =========================================================
# FASTA validation
# =========================================================

VALID_DNA_PATTERN = re.compile(
    r"^[ACGTNRYKMSWBDHV\-]+$"
)


def audit_fasta_file(fasta_path):

    fasta_records = 0
    total_sequence_bp = 0

    valid_sequence = True
    found_sequence = False


    with open(
        fasta_path,
        "rt",
        encoding="utf-8",
        errors="replace"
    ) as f:

        for raw_line in f:

            line = raw_line.strip()

            if not line:
                continue


            if line.startswith(">"):

                fasta_records += 1
                continue


            seq = line.upper()


            if not VALID_DNA_PATTERN.fullmatch(seq):

                valid_sequence = False


            total_sequence_bp += len(seq)

            found_sequence = True


    valid_sequence = (
        valid_sequence
        and found_sequence
        and fasta_records > 0
    )


    return {

        "fasta_records":
            fasta_records,

        "total_sequence_bp":
            total_sequence_bp,

        "valid_sequence":
            valid_sequence
    }


# =========================================================
# Download one genome directly from BV-BRC FTPS
# =========================================================

def download_genome(job):

    genome_id = job[
        "Genome ID"
    ]

    phenotype = job[
        "Phenotype"
    ]


    expected_length = int(
        qc_lookup.loc[
            genome_id,
            "genome_length"
        ]
    )


    expected_contigs = int(
        qc_lookup.loc[
            genome_id,
            "contigs"
        ]
    )


    # -----------------------------------------------------
    # BV-BRC direct genome FASTA location
    #
    # Explicit FTPS:
    # ftp:// + --ssl-reqd
    # -----------------------------------------------------

    ftp_url = (
        "ftp://ftp.bv-brc.org/"
        f"genomes/{genome_id}/{genome_id}.fna"
    )


    # -----------------------------------------------------
    # Temporary raw FASTA file
    # -----------------------------------------------------

    raw_temp_file = os.path.join(
        LOCAL_FASTA_DIR,
        f"{genome_id}.download.fna"
    )


    output_file = os.path.join(
        LOCAL_FASTA_DIR,
        f"{genome_id}.fna.gz"
    )


    gzip_temp_file = (
        output_file
        + ".tmp"
    )


    try:

        # -------------------------------------------------
        # Remove stale partial files from previous attempts
        # -------------------------------------------------

        for temp_path in [
            raw_temp_file,
            gzip_temp_file
        ]:

            if os.path.exists(
                temp_path
            ):

                os.remove(
                    temp_path
                )


        # -------------------------------------------------
        # Direct FTPS transfer using curl
        #
        # --ssl-reqd:
        #     require encrypted FTP connection
        #
        # --retry:
        #     retry transient failures
        #
        # --retry-all-errors:
        #     retry network/TLS failures as well
        # -------------------------------------------------

        curl_command = [

            "curl",

            "--fail",
            "--silent",
            "--show-error",

            "--ssl-reqd",

            "--user",
            "anonymous:guest",

            "--connect-timeout",
            "20",

            "--max-time",
            "180",

            "--retry",
            "4",

            "--retry-delay",
            "1",

            "--retry-all-errors",

            "--output",
            raw_temp_file,

            ftp_url
        ]


        transfer = subprocess.run(

            curl_command,

            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,

            text=True
        )


        # -------------------------------------------------
        # Check transfer result
        # -------------------------------------------------

        if transfer.returncode != 0:

            error_message = (
                transfer.stderr.strip()
                or
                f"curl exit code {transfer.returncode}"
            )

            raise RuntimeError(
                error_message
            )


        if not os.path.exists(
            raw_temp_file
        ):

            raise RuntimeError(
                "FTPS transfer finished but "
                "no FASTA file was created."
            )


        if os.path.getsize(
            raw_temp_file
        ) == 0:

            raise RuntimeError(
                "Downloaded FASTA file is empty."
            )


        # -------------------------------------------------
        # Audit downloaded FASTA
        # -------------------------------------------------

        fasta_audit = (
            audit_fasta_file(
                raw_temp_file
            )
        )


        fasta_records = (
            fasta_audit[
                "fasta_records"
            ]
        )


        fasta_length = (
            fasta_audit[
                "total_sequence_bp"
            ]
        )


        # -------------------------------------------------
        # Compare with QC metadata
        # -------------------------------------------------

        record_match = (
            fasta_records
            == expected_contigs
        )


        length_difference = (
            fasta_length
            - expected_length
        )


        length_match = (
            fasta_length
            == expected_length
        )


        complete_fasta = (

            fasta_audit[
                "valid_sequence"
            ]

            and record_match

            and length_match
        )


        compressed_size = 0


        # -------------------------------------------------
        # Compress ONLY validated assemblies
        # -------------------------------------------------

        if complete_fasta:

            with open(
                raw_temp_file,
                "rb"
            ) as src:

                with gzip.open(
                    gzip_temp_file,
                    "wb",
                    compresslevel=1
                ) as dst:

                    shutil.copyfileobj(
                        src,
                        dst,
                        length=1024 * 1024
                    )


            # Atomic move to final filename
            os.replace(
                gzip_temp_file,
                output_file
            )


            compressed_size = (
                os.path.getsize(
                    output_file
                )
            )


        # -------------------------------------------------
        # Remove temporary raw FASTA
        # -------------------------------------------------

        if os.path.exists(
            raw_temp_file
        ):

            os.remove(
                raw_temp_file
            )


        # -------------------------------------------------
        # Remove temporary gzip if validation failed
        # -------------------------------------------------

        if (
            not complete_fasta
            and os.path.exists(
                gzip_temp_file
            )
        ):

            os.remove(
                gzip_temp_file
            )


        # -------------------------------------------------
        # Return result
        #
        # "HTTP Status" retained only because Cell 6 audit
        # schema already expects this column.
        # -------------------------------------------------

        return {

            "Genome ID":
                genome_id,

            "Phenotype":
                phenotype,

            "HTTP Status":
                "FTPS_OK",

            "Expected Contigs":
                expected_contigs,

            "FASTA Records":
                fasta_records,

            "Record Count Match":
                record_match,

            "Expected Length (bp)":
                expected_length,

            "FASTA Length (bp)":
                fasta_length,

            "Length Difference (bp)":
                length_difference,

            "Length Match":
                length_match,

            "Valid Sequence":
                fasta_audit[
                    "valid_sequence"
                ],

            "Complete FASTA":
                complete_fasta,

            "Compressed Size (bytes)":
                compressed_size,

            "Output File":
                (
                    output_file
                    if complete_fasta
                    else None
                ),

            "Error":
                (
                    None
                    if complete_fasta
                    else
                    "Downloaded FASTA did not match "
                    "expected genome length and/or contig count."
                )
        }


    except Exception as e:

        # -------------------------------------------------
        # Clean temporary files after failure
        # -------------------------------------------------

        for temp_path in [
            raw_temp_file,
            gzip_temp_file
        ]:

            try:

                if os.path.exists(
                    temp_path
                ):

                    os.remove(
                        temp_path
                    )

            except Exception:

                pass


        return {

            "Genome ID":
                genome_id,

            "Phenotype":
                phenotype,

            "HTTP Status":
                None,

            "Expected Contigs":
                expected_contigs,

            "FASTA Records":
                None,

            "Record Count Match":
                False,

            "Expected Length (bp)":
                expected_length,

            "FASTA Length (bp)":
                None,

            "Length Difference (bp)":
                None,

            "Length Match":
                False,

            "Valid Sequence":
                False,

            "Complete FASTA":
                False,

            "Compressed Size (bytes)":
                0,

            "Output File":
                None,

            "Error":
                str(e)
        }

# Cell 6 — Parallel Full-Cohort FASTA Retrieval

In [ ]:
import os
import csv
import time
import shutil
import subprocess

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)


# =========================================================
# 1. Audit schema
# =========================================================

# Keep the same schema used previously so old API-based
# retrieval records and new FTPS-based records remain
# compatible in the same audit file.

audit_columns = [

    "Genome ID",
    "Phenotype",
    "HTTP Status",

    "Expected Contigs",
    "FASTA Records",
    "Record Count Match",

    "Expected Length (bp)",
    "FASTA Length (bp)",
    "Length Difference (bp)",
    "Length Match",

    "Valid Sequence",
    "Complete FASTA",

    "Compressed Size (bytes)",
    "Output File",

    "Error"
]


# =========================================================
# 2. Detect already completed FASTA files
# =========================================================

existing_files = {

    filename[:-7]

    for filename in os.listdir(
        LOCAL_FASTA_DIR
    )

    if filename.endswith(
        ".fna.gz"
    )
}


completed_ids = (
    existing_files
    & manifest_ids
)


# =========================================================
# 3. Prepare remaining retrieval jobs
# =========================================================

jobs = []


for _, row in df_manifest_qc.iterrows():

    genome_id = str(
        row["Genome ID String"]
    ).strip()


    if genome_id in completed_ids:
        continue


    jobs.append({

        "Genome ID":
            genome_id,

        "Phenotype":
            row[
                "Resistant Phenotype"
            ]
    })


print("Retrieval status")
print("-" * 60)

print(
    f"Already completed : "
    f"{len(completed_ids):,}"
)

print(
    f"Remaining genomes : "
    f"{len(jobs):,}"
)

print(
    f"Parallel workers  : "
    f"{MAX_WORKERS}"
)


# =========================================================
# 4. Persistent backup helper
# =========================================================

def backup_progress():

    backup_ok = True


    # -----------------------------------------------------
    # Backup audit file
    # -----------------------------------------------------

    try:

        if os.path.exists(
            LOCAL_AUDIT_PATH
        ):

            shutil.copy2(
                LOCAL_AUDIT_PATH,
                DRIVE_AUDIT_PATH
            )


    except Exception as e:

        backup_ok = False

        print(
            f"WARNING: Audit backup failed: {e}"
        )


    # -----------------------------------------------------
    # Backup ONLY final validated .fna.gz files
    #
    # Temporary files such as:
    #   *.download.fna
    #   *.tmp
    #
    # are explicitly excluded.
    # -----------------------------------------------------

    try:

        result = subprocess.run(

            [
                "rsync",

                "-a",

                "--ignore-existing",

                "--include=*.fna.gz",
                "--exclude=*",

                LOCAL_FASTA_DIR + "/",
                DRIVE_FASTA_DIR + "/"
            ],

            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,

            text=True
        )


        if result.returncode != 0:

            backup_ok = False

            print(
                "WARNING: FASTA backup failed:"
            )

            print(
                result.stderr.strip()
            )


    except Exception as e:

        backup_ok = False

        print(
            f"WARNING: FASTA backup failed: {e}"
        )


    return backup_ok


# =========================================================
# 5. Skip retrieval entirely if everything is already done
# =========================================================

if len(jobs) == 0:

    print()
    print(
        "No remaining genomes to retrieve."
    )

    print(
        "All QC-qualified genomes already "
        "have FASTA files."
    )


# =========================================================
# 6. Parallel FTPS retrieval
# =========================================================

else:

    new_results = []

    start_time = time.time()


    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:


        futures = {

            executor.submit(
                download_genome,
                job
            ):
            job["Genome ID"]

            for job in jobs
        }


        for number_completed, future in enumerate(

            as_completed(
                futures
            ),

            start=1
        ):


            genome_id = futures[
                future
            ]


            # -------------------------------------------------
            # download_genome() should already catch errors,
            # but keep this protection in case a worker itself
            # crashes unexpectedly.
            # -------------------------------------------------

            try:

                result = future.result()


            except Exception as e:

                result = {

                    "Genome ID":
                        genome_id,

                    "Phenotype":
                        None,

                    "HTTP Status":
                        None,

                    "Expected Contigs":
                        None,

                    "FASTA Records":
                        None,

                    "Record Count Match":
                        False,

                    "Expected Length (bp)":
                        None,

                    "FASTA Length (bp)":
                        None,

                    "Length Difference (bp)":
                        None,

                    "Length Match":
                        False,

                    "Valid Sequence":
                        False,

                    "Complete FASTA":
                        False,

                    "Compressed Size (bytes)":
                        0,

                    "Output File":
                        None,

                    "Error":
                        f"Unhandled worker error: {e}"
                }


            new_results.append(
                result
            )


            # =================================================
            # 7. Immediate LOCAL checkpoint
            # =================================================

            file_exists = os.path.exists(
                LOCAL_AUDIT_PATH
            )


            with open(

                LOCAL_AUDIT_PATH,

                "a",

                newline="",

                encoding="utf-8"

            ) as f:


                writer = csv.DictWriter(

                    f,

                    fieldnames=audit_columns
                )


                if not file_exists:

                    writer.writeheader()


                writer.writerow(
                    result
                )


            # =================================================
            # 8. Progress report every 50 genomes
            # =================================================

            if (
                number_completed % 50 == 0
                or number_completed == len(jobs)
            ):


                successes = sum(

                    bool(
                        r["Complete FASTA"]
                    )

                    for r in new_results
                )


                failures = (
                    number_completed
                    - successes
                )


                elapsed_minutes = (

                    time.time()
                    - start_time

                ) / 60


                rate = (

                    number_completed
                    / elapsed_minutes

                    if elapsed_minutes > 0

                    else 0
                )


                remaining = (

                    len(jobs)
                    - number_completed
                )


                eta_minutes = (

                    remaining
                    / rate

                    if rate > 0

                    else float("nan")
                )


                print(

                    f"[{number_completed:4d}/{len(jobs)}] "

                    f"successful: "
                    f"{successes:,} | "

                    f"failed: "
                    f"{failures:,} | "

                    f"rate: "
                    f"{rate:.1f} genomes/min | "

                    f"ETA: "
                    f"{eta_minutes:.1f} min"
                )


            # =================================================
            # 9. Persistent Google Drive backup
            # =================================================

            if (
                number_completed
                % BACKUP_EVERY
                == 0
            ):


                print()
                print(
                    "Backing up progress "
                    "to Google Drive..."
                )


                backup_ok = (
                    backup_progress()
                )


                if backup_ok:

                    print(
                        "Backup complete."
                    )

                else:

                    print(
                        "Backup incomplete, but retrieval "
                        "will continue."
                    )


                print()


    # =========================================================
    # 10. Final backup after retrieval loop
    # =========================================================

    print()
    print(
        "Retrieval loop finished."
    )


    print(
        "Running final backup..."
    )


    final_backup_ok = (
        backup_progress()
    )


    if final_backup_ok:

        print(
            "Final checkpoint backed up "
            "to Google Drive."
        )

    else:

        print(
            "WARNING: Final Drive backup was incomplete."
        )

        print(
            "Local FASTA files and local audit "
            "remain available in /content."
        )

Retrieval status
------------------------------------------------------------
Already completed : 1,236
Remaining genomes : 2,997
Parallel workers  : 8
[  50/2997] successful: 50 | failed: 0 | rate: 14.0 genomes/min | ETA: 211.2 min
[ 100/2997] successful: 100 | failed: 0 | rate: 14.2 genomes/min | ETA: 204.2 min
[ 150/2997] successful: 150 | failed: 0 | rate: 14.1 genomes/min | ETA: 202.0 min
[ 200/2997] successful: 200 | failed: 0 | rate: 14.3 genomes/min | ETA: 196.1 min

Backing up progress to Google Drive...
Backup complete.

[ 250/2997] successful: 250 | failed: 0 | rate: 13.7 genomes/min | ETA: 201.0 min
[ 300/2997] successful: 300 | failed: 0 | rate: 13.6 genomes/min | ETA: 197.9 min
[ 350/2997] successful: 350 | failed: 0 | rate: 13.7 genomes/min | ETA: 193.5 min
[ 400/2997] successful: 400 | failed: 0 | rate: 13.7 genomes/min | ETA: 189.4 min

Backing up progress to Google Drive...
Backup complete.

[ 450/2997] successful: 450 | failed: 0 | rate: 13.8 genomes/min | ETA: 184.4

# Cell 7 — Final FASTA Retrieval Integrity Audit

In [ ]:
import os
import shutil
import subprocess
import pandas as pd


# =========================================================
# 1. Load retrieval audit
# =========================================================

df_fasta_audit = pd.read_csv(

    LOCAL_AUDIT_PATH,

    dtype={
        "Genome ID": str
    },

    on_bad_lines="skip"
)


# Normalize Genome IDs
df_fasta_audit[
    "Genome ID"
] = (

    df_fasta_audit[
        "Genome ID"
    ]

    .astype(str)
    .str.strip()
)


# Normalize Complete FASTA column
df_fasta_audit[
    "Complete FASTA"
] = (

    df_fasta_audit[
        "Complete FASTA"
    ]

    .astype(str)
    .str.lower()

    .map({

        "true":
            True,

        "false":
            False
    })
)


# =========================================================
# 2. Keep latest retrieval attempt per genome
# =========================================================

df_fasta_audit = (

    df_fasta_audit

    .drop_duplicates(

        subset=[
            "Genome ID"
        ],

        keep="last"
    )

    .reset_index(
        drop=True
    )
)


# Save cleaned audit locally
df_fasta_audit.to_csv(

    LOCAL_AUDIT_PATH,

    index=False
)


# =========================================================
# 3. Detect actual validated FASTA files
# =========================================================

saved_files = {

    filename[:-7]

    for filename in os.listdir(
        LOCAL_FASTA_DIR
    )

    if filename.endswith(
        ".fna.gz"
    )
}


saved_manifest_ids = (

    saved_files
    & manifest_ids
)


missing_fasta_ids = (

    manifest_ids
    - saved_manifest_ids
)


unexpected_fasta_ids = (

    saved_files
    - manifest_ids
)


# =========================================================
# 4. Calculate storage size
# =========================================================

compressed_gib = (

    sum(

        os.path.getsize(

            os.path.join(

                LOCAL_FASTA_DIR,

                f"{genome_id}.fna.gz"
            )
        )

        for genome_id
        in saved_manifest_ids
    )

    / (1024 ** 3)
)


# =========================================================
# 5. Audit retrieval records
# =========================================================

audit_manifest_ids = set(

    df_fasta_audit[
        "Genome ID"
    ]
)


missing_audit_ids = (

    manifest_ids
    - audit_manifest_ids
)


complete_audit_ids = set(

    df_fasta_audit.loc[

        df_fasta_audit[
            "Complete FASTA"
        ] == True,

        "Genome ID"
    ]
)


# =========================================================
# 6. Final retrieval report
# =========================================================

print()
print("=" * 70)

print(
    "FULL FASTA RETRIEVAL SUMMARY"
)

print("=" * 70)


print(
    f"Expected genomes          : "
    f"{len(manifest_ids):,}"
)


print(
    f"Validated FASTA files     : "
    f"{len(saved_manifest_ids):,}"
)


print(
    f"Missing / incomplete      : "
    f"{len(missing_fasta_ids):,}"
)


print(
    f"Unexpected FASTA files    : "
    f"{len(unexpected_fasta_ids):,}"
)


print(
    f"Genome IDs in audit       : "
    f"{len(audit_manifest_ids):,}"
)


print(
    f"Audit IDs marked complete : "
    f"{len(complete_audit_ids):,}"
)


print(
    f"Missing audit records     : "
    f"{len(missing_audit_ids):,}"
)


print(
    f"Compressed storage        : "
    f"{compressed_gib:.2f} GiB"
)


# =========================================================
# 7. PASS / WARNING
# =========================================================

retrieval_pass = (

    len(saved_manifest_ids)
    == len(manifest_ids)

    and len(missing_fasta_ids)
    == 0

    and len(unexpected_fasta_ids)
    == 0
)


if retrieval_pass:

    print()

    print(
        "PASS: Complete validated FASTA assemblies "
        "are available for all 4,233 QC-qualified genomes."
    )


else:

    print()

    print(
        "WARNING: Full FASTA retrieval is not yet complete."
    )


    if len(missing_fasta_ids) > 0:

        print()

        print(
            "First 20 missing Genome IDs:"
        )

        print(
            sorted(
                missing_fasta_ids
            )[:20]
        )


    if len(unexpected_fasta_ids) > 0:

        print()

        print(
            "First 20 unexpected FASTA Genome IDs:"
        )

        print(
            sorted(
                unexpected_fasta_ids
            )[:20]
        )


    if len(missing_audit_ids) > 0:

        print()

        print(
            "First 20 Genome IDs missing from audit:"
        )

        print(
            sorted(
                missing_audit_ids
            )[:20]
        )


# =========================================================
# 8. Backup final audit to Google Drive
# =========================================================

print()
print(
    "Backing up final retrieval audit..."
)


shutil.copy2(

    LOCAL_AUDIT_PATH,

    DRIVE_AUDIT_PATH
)


# =========================================================
# 9. Backup ONLY final validated FASTA files
#
# Exclude temporary files such as:
#   *.download.fna
#   *.tmp
# =========================================================

print(
    "Backing up validated FASTA files..."
)


backup_result = subprocess.run(

    [
        "rsync",

        "-a",

        "--ignore-existing",

        "--include=*.fna.gz",

        "--exclude=*",

        LOCAL_FASTA_DIR + "/",

        DRIVE_FASTA_DIR + "/"
    ],

    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,

    text=True
)


if backup_result.returncode == 0:

    print(
        "Final retrieval audit and FASTA files "
        "backed up to Google Drive."
    )

else:

    print(
        "WARNING: Final Google Drive backup "
        "did not complete successfully."
    )

    print(
        backup_result.stderr.strip()
    )


# =========================================================
# 10. Confirm backed-up FASTA count
# =========================================================

drive_fasta_files = {

    filename[:-7]

    for filename in os.listdir(
        DRIVE_FASTA_DIR
    )

    if filename.endswith(
        ".fna.gz"
    )
}


drive_manifest_ids = (

    drive_fasta_files
    & manifest_ids
)


print()
print(
    f"Validated FASTA files on Google Drive : "
    f"{len(drive_manifest_ids):,}"
)


if (
    retrieval_pass
    and len(drive_manifest_ids)
    == len(manifest_ids)
):

    print(
        "\nFINAL PASS: All 4,233 validated genome "
        "assemblies are present locally and backed up "
        "to Google Drive."
    )

elif retrieval_pass:

    print(
        "\nLOCAL PASS, BACKUP WARNING: "
        "All genome assemblies are available locally, "
        "but the Google Drive backup is incomplete."
    )


FULL FASTA RETRIEVAL SUMMARY
Expected genomes          : 4,233
Validated FASTA files     : 4,233
Missing / incomplete      : 0
Unexpected FASTA files    : 0
Genome IDs in audit       : 3,484
Audit IDs marked complete : 3,475
Missing audit records     : 749
Compressed storage        : 7.71 GiB

PASS: Complete validated FASTA assemblies are available for all 4,233 QC-qualified genomes.

Backing up final retrieval audit...
Backing up validated FASTA files...
Final retrieval audit and FASTA files backed up to Google Drive.

Validated FASTA files on Google Drive : 4,233

FINAL PASS: All 4,233 validated genome assemblies are present locally and backed up to Google Drive.


# Cell 8 — Reconstruct and Complete FASTA Retrieval Audit

In [ ]:
import os
import re
import gzip
import shutil
import pandas as pd


# =========================================================
# 1. Use persistent Google Drive storage
# =========================================================

FASTA_DIR = DRIVE_FASTA_DIR
AUDIT_PATH = DRIVE_AUDIT_PATH


assert os.path.isdir(
    FASTA_DIR
), f"FASTA directory not found: {FASTA_DIR}"

assert os.path.exists(
    AUDIT_PATH
), f"Audit file not found: {AUDIT_PATH}"


# =========================================================
# 2. Confirm FASTA files on Google Drive
# =========================================================

drive_fasta_ids = {

    filename[:-7]

    for filename in os.listdir(
        FASTA_DIR
    )

    if filename.endswith(
        ".fna.gz"
    )
}


drive_manifest_ids = (
    drive_fasta_ids
    & manifest_ids
)


missing_drive_fastas = (
    manifest_ids
    - drive_manifest_ids
)


unexpected_drive_fastas = (
    drive_fasta_ids
    - manifest_ids
)


print("Google Drive FASTA status")
print("-" * 60)

print(
    f"Expected genomes          : "
    f"{len(manifest_ids):,}"
)

print(
    f"FASTA files on Drive      : "
    f"{len(drive_manifest_ids):,}"
)

print(
    f"Missing FASTA files       : "
    f"{len(missing_drive_fastas):,}"
)

print(
    f"Unexpected FASTA files    : "
    f"{len(unexpected_drive_fastas):,}"
)


assert len(missing_drive_fastas) == 0, (
    f"{len(missing_drive_fastas)} expected FASTA files "
    f"are missing from Google Drive."
)


# =========================================================
# 3. Load current retrieval audit from Google Drive
# =========================================================

df_fasta_audit = pd.read_csv(
    AUDIT_PATH,
    dtype={
        "Genome ID": str
    },
    on_bad_lines="skip"
)


df_fasta_audit["Genome ID"] = (
    df_fasta_audit["Genome ID"]
    .astype(str)
    .str.strip()
)


# Normalize Complete FASTA
df_fasta_audit["Complete FASTA"] = (
    df_fasta_audit["Complete FASTA"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)


# Keep latest record per genome
df_fasta_audit = (
    df_fasta_audit
    .drop_duplicates(
        subset=["Genome ID"],
        keep="last"
    )
    .reset_index(drop=True)
)


# =========================================================
# 4. Back up the pre-reconstruction audit
# =========================================================

audit_backup_path = (
    AUDIT_PATH
    + ".before_reconstruction.csv"
)


if not os.path.exists(
    audit_backup_path
):

    shutil.copy2(
        AUDIT_PATH,
        audit_backup_path
    )

    print()
    print(
        "Original audit backed up before reconstruction."
    )


# =========================================================
# 5. Identify genomes requiring audit reconstruction
# =========================================================

audit_ids = set(
    df_fasta_audit[
        "Genome ID"
    ]
)


complete_audit_ids = set(
    df_fasta_audit.loc[
        df_fasta_audit[
            "Complete FASTA"
        ] == True,
        "Genome ID"
    ]
)


missing_audit_ids = (
    manifest_ids
    - audit_ids
)


incomplete_audit_ids = (
    manifest_ids
    - complete_audit_ids
)


# Includes:
# - completely missing audit records
# - existing records not marked Complete FASTA = True
repair_ids = sorted(
    incomplete_audit_ids
)


print()
print("Audit reconstruction status")
print("-" * 60)

print(
    f"Expected genomes                 : "
    f"{len(manifest_ids):,}"
)

print(
    f"Genome IDs currently in audit    : "
    f"{len(audit_ids):,}"
)

print(
    f"Audit IDs currently complete     : "
    f"{len(complete_audit_ids):,}"
)

print(
    f"Completely missing audit records : "
    f"{len(missing_audit_ids):,}"
)

print(
    f"Total genomes requiring re-audit : "
    f"{len(repair_ids):,}"
)


# =========================================================
# 6. FASTA validation helper
# =========================================================

VALID_DNA_PATTERN = re.compile(
    r"^[ACGTNRYKMSWBDHV\-]+$"
)


def audit_gzip_fasta(
    fasta_gz_path
):

    fasta_records = 0
    total_sequence_bp = 0

    valid_sequence = True
    found_sequence = False


    with gzip.open(
        fasta_gz_path,
        "rt",
        encoding="utf-8",
        errors="replace"
    ) as f:


        for raw_line in f:

            line = raw_line.strip()


            if not line:
                continue


            if line.startswith(">"):

                fasta_records += 1
                continue


            sequence = line.upper()


            if not VALID_DNA_PATTERN.fullmatch(
                sequence
            ):

                valid_sequence = False


            total_sequence_bp += len(
                sequence
            )


            found_sequence = True


    valid_sequence = (
        valid_sequence
        and found_sequence
        and fasta_records > 0
    )


    return {

        "fasta_records":
            fasta_records,

        "total_sequence_bp":
            total_sequence_bp,

        "valid_sequence":
            valid_sequence
    }


# =========================================================
# 7. Phenotype lookup
# =========================================================

phenotype_lookup = (
    df_manifest_qc
    .set_index(
        "Genome ID String"
    )[
        "Resistant Phenotype"
    ]
    .to_dict()
)


# =========================================================
# 8. Re-audit missing / incomplete records from Drive
# =========================================================

reconstructed_records = []


for i, genome_id in enumerate(
    repair_ids,
    start=1
):


    fasta_path = os.path.join(
        FASTA_DIR,
        f"{genome_id}.fna.gz"
    )


    expected_length = int(
        qc_lookup.loc[
            genome_id,
            "genome_length"
        ]
    )


    expected_contigs = int(
        qc_lookup.loc[
            genome_id,
            "contigs"
        ]
    )


    try:

        if not os.path.exists(
            fasta_path
        ):

            raise FileNotFoundError(
                f"FASTA file not found: {fasta_path}"
            )


        fasta_audit = (
            audit_gzip_fasta(
                fasta_path
            )
        )


        fasta_records = (
            fasta_audit[
                "fasta_records"
            ]
        )


        fasta_length = (
            fasta_audit[
                "total_sequence_bp"
            ]
        )


        record_match = (
            fasta_records
            == expected_contigs
        )


        length_difference = (
            fasta_length
            - expected_length
        )


        length_match = (
            fasta_length
            == expected_length
        )


        complete_fasta = (

            fasta_audit[
                "valid_sequence"
            ]

            and record_match

            and length_match
        )


        reconstructed_records.append({

            "Genome ID":
                genome_id,

            "Phenotype":
                phenotype_lookup[
                    genome_id
                ],

            "HTTP Status":
                "DRIVE_REAUDIT",

            "Expected Contigs":
                expected_contigs,

            "FASTA Records":
                fasta_records,

            "Record Count Match":
                record_match,

            "Expected Length (bp)":
                expected_length,

            "FASTA Length (bp)":
                fasta_length,

            "Length Difference (bp)":
                length_difference,

            "Length Match":
                length_match,

            "Valid Sequence":
                fasta_audit[
                    "valid_sequence"
                ],

            "Complete FASTA":
                complete_fasta,

            "Compressed Size (bytes)":
                os.path.getsize(
                    fasta_path
                ),

            "Output File":
                fasta_path,

            "Error":
                (
                    None
                    if complete_fasta
                    else
                    "Drive FASTA failed QC re-audit."
                )
        })


    except Exception as e:

        reconstructed_records.append({

            "Genome ID":
                genome_id,

            "Phenotype":
                phenotype_lookup.get(
                    genome_id
                ),

            "HTTP Status":
                "DRIVE_REAUDIT",

            "Expected Contigs":
                expected_contigs,

            "FASTA Records":
                None,

            "Record Count Match":
                False,

            "Expected Length (bp)":
                expected_length,

            "FASTA Length (bp)":
                None,

            "Length Difference (bp)":
                None,

            "Length Match":
                False,

            "Valid Sequence":
                False,

            "Complete FASTA":
                False,

            "Compressed Size (bytes)":
                0,

            "Output File":
                None,

            "Error":
                str(e)
        })


    # Progress every 50 because Drive I/O may be slower
    if (
        i % 50 == 0
        or i == len(repair_ids)
    ):


        reconstructed_success = sum(

            bool(
                record[
                    "Complete FASTA"
                ]
            )

            for record
            in reconstructed_records
        )


        print(

            f"[{i:4d}/{len(repair_ids)}] "

            f"re-audited | "

            f"complete: "
            f"{reconstructed_success:,}"
        )


# =========================================================
# 9. Merge reconstructed records
# =========================================================

if reconstructed_records:


    df_reconstructed = pd.DataFrame(
        reconstructed_records
    )


    df_fasta_audit = pd.concat(

        [
            df_fasta_audit,
            df_reconstructed
        ],

        ignore_index=True
    )


# Keep latest record per genome
df_fasta_audit = (
    df_fasta_audit
    .drop_duplicates(
        subset=["Genome ID"],
        keep="last"
    )
    .reset_index(drop=True)
)


# Normalize boolean one final time
df_fasta_audit[
    "Complete FASTA"
] = (
    df_fasta_audit[
        "Complete FASTA"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)


# =========================================================
# 10. Final audit integrity checks
# =========================================================

final_audit_ids = set(
    df_fasta_audit[
        "Genome ID"
    ]
)


final_complete_ids = set(
    df_fasta_audit.loc[
        df_fasta_audit[
            "Complete FASTA"
        ] == True,
        "Genome ID"
    ]
)


final_missing_ids = (
    manifest_ids
    - final_audit_ids
)


final_incomplete_ids = (
    manifest_ids
    - final_complete_ids
)


unexpected_audit_ids = (
    final_audit_ids
    - manifest_ids
)


# =========================================================
# 11. Save completed audit DIRECTLY to Google Drive
# =========================================================

df_fasta_audit.to_csv(
    AUDIT_PATH,
    index=False
)


# =========================================================
# 12. Final report
# =========================================================

print()
print("=" * 70)

print(
    "COMPLETED FASTA RETRIEVAL AUDIT"
)

print("=" * 70)


print(
    f"Expected genomes          : "
    f"{len(manifest_ids):,}"
)


print(
    f"Genome IDs in audit       : "
    f"{len(final_audit_ids):,}"
)


print(
    f"Audit IDs marked complete : "
    f"{len(final_complete_ids):,}"
)


print(
    f"Missing audit records     : "
    f"{len(final_missing_ids):,}"
)


print(
    f"Incomplete audit records  : "
    f"{len(final_incomplete_ids):,}"
)


print(
    f"Unexpected audit IDs      : "
    f"{len(unexpected_audit_ids):,}"
)


# =========================================================
# 13. PASS / WARNING
# =========================================================

audit_pass = (

    len(final_audit_ids)
    == len(manifest_ids)

    and len(final_complete_ids)
    == len(manifest_ids)

    and len(final_missing_ids)
    == 0

    and len(final_incomplete_ids)
    == 0

    and len(unexpected_audit_ids)
    == 0
)


if audit_pass:

    print()

    print(
        "PASS: The FASTA retrieval audit is complete "
        "for all 4,233 QC-qualified genomes."
    )

    print(
        "All assemblies have verified contig counts "
        "and genome lengths matching the QC reference."
    )

    print(
        "The completed audit has been saved "
        "persistently to Google Drive."
    )


else:

    print()

    print(
        "WARNING: FASTA retrieval audit still "
        "contains unresolved records."
    )


    if final_missing_ids:

        print()

        print(
            "First 20 missing audit Genome IDs:"
        )

        print(
            sorted(
                final_missing_ids
            )[:20]
        )


    if final_incomplete_ids:

        print()

        print(
            "First 20 incomplete audit Genome IDs:"
        )

        print(
            sorted(
                final_incomplete_ids
            )[:20]
        )


    unresolved_records = (
        df_fasta_audit[
            df_fasta_audit[
                "Genome ID"
            ].isin(
                final_incomplete_ids
            )
        ]
    )


    if len(
        unresolved_records
    ) > 0:

        print()

        print(
            "Unresolved audit records:"
        )

        display(
            unresolved_records[
                [
                    "Genome ID",
                    "Phenotype",
                    "Expected Contigs",
                    "FASTA Records",
                    "Expected Length (bp)",
                    "FASTA Length (bp)",
                    "Error"
                ]
            ].head(20)
        )

Google Drive FASTA status
------------------------------------------------------------
Expected genomes          : 4,233
FASTA files on Drive      : 4,233
Missing FASTA files       : 0
Unexpected FASTA files    : 0

Original audit backed up before reconstruction.

Audit reconstruction status
------------------------------------------------------------
Expected genomes                 : 4,233
Genome IDs currently in audit    : 3,484
Audit IDs currently complete     : 3,475
Completely missing audit records : 749
Total genomes requiring re-audit : 758
[  50/758] re-audited | complete: 50
[ 100/758] re-audited | complete: 100
[ 150/758] re-audited | complete: 150
[ 200/758] re-audited | complete: 200
[ 250/758] re-audited | complete: 250
[ 300/758] re-audited | complete: 300
[ 350/758] re-audited | complete: 350
[ 400/758] re-audited | complete: 400
[ 450/758] re-audited | complete: 450
[ 500/758] re-audited | complete: 500
[ 550/758] re-audited | complete: 550
[ 600/758] re-audited | comp

# Cell 9 — Create Genomic Feature Engineering Input Manifest


In [ ]:
import os
import pandas as pd


# =========================================================
# 1. Output path
# =========================================================

FEATURE_INPUT_DIR = os.path.join(
    DRIVE_ROOT,
    "data/processed"
)

os.makedirs(
    FEATURE_INPUT_DIR,
    exist_ok=True
)

FEATURE_INPUT_MANIFEST_PATH = os.path.join(
    FEATURE_INPUT_DIR,
    "genomic_feature_input_manifest.csv"
)


# =========================================================
# 2. Load completed FASTA retrieval audit
# =========================================================

df_fasta_audit_final = pd.read_csv(
    DRIVE_AUDIT_PATH,
    dtype={
        "Genome ID": str
    }
)


df_fasta_audit_final["Genome ID"] = (
    df_fasta_audit_final["Genome ID"]
    .astype(str)
    .str.strip()
)


df_fasta_audit_final["Complete FASTA"] = (
    df_fasta_audit_final["Complete FASTA"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)


# Keep latest record per genome
df_fasta_audit_final = (
    df_fasta_audit_final
    .drop_duplicates(
        subset=["Genome ID"],
        keep="last"
    )
    .reset_index(drop=True)
)


# =========================================================
# 3. Prepare QC-qualified phenotype manifest
# =========================================================

df_feature_input = (
    df_manifest_qc[
        [
            "Genome ID String",
            "Genome Name",
            "Taxon ID",
            "Resistant Phenotype"
        ]
    ]
    .copy()
)


df_feature_input["Genome ID String"] = (
    df_feature_input["Genome ID String"]
    .astype(str)
    .str.strip()
)


df_feature_input = df_feature_input.rename(
    columns={
        "Genome ID String": "Genome ID"
    }
)


# =========================================================
# 4. Add QC reference values
# =========================================================

df_qc_for_features = (
    qc_lookup[
        [
            "genome_length",
            "contigs"
        ]
    ]
    .reset_index()
    .rename(
        columns={
            qc_lookup.index.name: "Genome ID"
        }
    )
)


df_qc_for_features["Genome ID"] = (
    df_qc_for_features["Genome ID"]
    .astype(str)
    .str.strip()
)


df_feature_input = (
    df_feature_input
    .merge(
        df_qc_for_features,
        on="Genome ID",
        how="left",
        validate="one_to_one"
    )
)


# =========================================================
# 5. Add FASTA audit status
# =========================================================

df_audit_for_features = (
    df_fasta_audit_final[
        [
            "Genome ID",
            "Complete FASTA"
        ]
    ]
    .copy()
)


df_feature_input = (
    df_feature_input
    .merge(
        df_audit_for_features,
        on="Genome ID",
        how="left",
        validate="one_to_one"
    )
)


# =========================================================
# 6. Construct persistent FASTA path
# =========================================================

df_feature_input["FASTA Path"] = (
    df_feature_input["Genome ID"]
    .apply(
        lambda genome_id:
            os.path.join(
                DRIVE_FASTA_DIR,
                f"{genome_id}.fna.gz"
            )
    )
)


# =========================================================
# 7. Integrity checks
# =========================================================

assert len(df_feature_input) == 4233, (
    f"Expected 4,233 genomes, "
    f"found {len(df_feature_input):,}."
)


assert (
    df_feature_input["Genome ID"].nunique()
    == 4233
), "Duplicate Genome IDs detected."


assert (
    df_feature_input["Genome ID"].notna().all()
), "Missing Genome IDs detected."


assert (
    df_feature_input["Resistant Phenotype"]
    .isin(
        [
            "Resistant",
            "Susceptible"
        ]
    )
    .all()
), "Unexpected phenotype detected."


assert (
    df_feature_input["genome_length"]
    .notna()
    .all()
), "Missing genome length metadata."


assert (
    df_feature_input["contigs"]
    .notna()
    .all()
), "Missing contig metadata."


assert (
    df_feature_input["Complete FASTA"]
    .eq(True)
    .all()
), "At least one genome lacks a complete FASTA audit."


# =========================================================
# 8. Final column order
# =========================================================

df_feature_input = df_feature_input[
    [
        "Genome ID",
        "Genome Name",
        "Taxon ID",
        "Resistant Phenotype",
        "genome_length",
        "contigs",
        "Complete FASTA",
        "FASTA Path"
    ]
]


# =========================================================
# 9. Save feature-engineering manifest
# =========================================================

df_feature_input.to_csv(
    FEATURE_INPUT_MANIFEST_PATH,
    index=False
)


# =========================================================
# 10. Summary
# =========================================================

print("=" * 70)
print("GENOMIC FEATURE ENGINEERING INPUT MANIFEST")
print("=" * 70)

print(
    f"Total genomes          : "
    f"{len(df_feature_input):,}"
)

print(
    f"Unique Genome IDs      : "
    f"{df_feature_input['Genome ID'].nunique():,}"
)

print(
    f"Complete FASTA genomes : "
    f"{df_feature_input['Complete FASTA'].sum():,}"
)

print()

print("Phenotype distribution:")

print(
    df_feature_input[
        "Resistant Phenotype"
    ]
    .value_counts()
)

print()

print(
    f"Saved to:"
    f"\n{FEATURE_INPUT_MANIFEST_PATH}"
)

print()

display(
    df_feature_input.head()
)

GENOMIC FEATURE ENGINEERING INPUT MANIFEST
Total genomes          : 4,233
Unique Genome IDs      : 4,233
Complete FASTA genomes : 4,233

Phenotype distribution:
Resistant Phenotype
Susceptible    2630
Resistant      1603
Name: count, dtype: int64

Saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/processed/genomic_feature_input_manifest.csv



,Genome ID,Genome Name,Taxon ID,Resistant Phenotype,genome_length,contigs,Complete FASTA,FASTA Path
0,1284787.3,Klebsiella pneumoniae UHKPC40,1284787,Resistant,5490156,155,True,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
1,1284788.3,Klebsiella pneumoniae UHKPC23,1284788,Resistant,5503448,195,True,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
2,1284789.4,Klebsiella pneumoniae UHKPC28,1284789,Resistant,5398963,137,True,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
3,1284790.3,Klebsiella pneumoniae UHKPC05,1284790,Resistant,5502006,187,True,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
4,1284791.4,Klebsiella pneumoniae UHKPC47,1284791,Resistant,5498288,195,True,/content/drive/MyDrive/AMR-Genome-ML/data/geno...


## Cell 10 — Install and Verify AMRFinderPlus

In [ ]:
import os
import subprocess
import textwrap


# =========================================================
# 1. Tool installation paths
# =========================================================

MICROMAMBA_ROOT = "/content/micromamba"
MICROMAMBA_BIN = os.path.join(
    MICROMAMBA_ROOT,
    "bin",
    "micromamba"
)

AMRFINDER_ENV = "/content/amrfinder_env"

AMRFINDER_BIN = os.path.join(
    AMRFINDER_ENV,
    "bin",
    "amrfinder"
)

AMRFINDER_UPDATE_BIN = os.path.join(
    AMRFINDER_ENV,
    "bin",
    "amrfinder_update"
)


# =========================================================
# 2. Install micromamba if necessary
# =========================================================

if not os.path.exists(MICROMAMBA_BIN):

    print("Installing micromamba...")

    os.makedirs(
        MICROMAMBA_ROOT,
        exist_ok=True
    )

    install_command = f"""
    set -e

    curl -Ls \
        https://micro.mamba.pm/api/micromamba/linux-64/latest \
        | tar -xj -C {MICROMAMBA_ROOT} bin/micromamba
    """

    subprocess.run(
        install_command,
        shell=True,
        check=True,
        executable="/bin/bash"
    )

else:

    print(
        "micromamba already installed."
    )


# =========================================================
# 3. Install AMRFinderPlus environment if necessary
# =========================================================

if not os.path.exists(AMRFINDER_BIN):

    print()
    print(
        "Installing NCBI AMRFinderPlus..."
    )

    subprocess.run(
        [
            MICROMAMBA_BIN,
            "create",
            "-y",
            "-p",
            AMRFINDER_ENV,

            "-c",
            "conda-forge",

            "-c",
            "bioconda",

            "ncbi-amrfinderplus"
        ],
        check=True
    )

else:

    print()
    print(
        "AMRFinderPlus already installed."
    )


# =========================================================
# 4. Verify executable
# =========================================================

assert os.path.exists(
    AMRFINDER_BIN
), (
    "AMRFinderPlus executable was not found."
)


# =========================================================
# 5. Print AMRFinderPlus version
# =========================================================

version_result = subprocess.run(
    [
        AMRFINDER_BIN,
        "--version"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True
)


print()
print("=" * 70)
print("AMRFINDERPLUS TOOL CHECK")
print("=" * 70)

print(
    version_result.stdout.strip()
)


# =========================================================
# 6. Confirm required companion executables
# =========================================================

required_tools = [
    "blastn",
    "blastp",
    "hmmsearch"
]


missing_tools = []


for tool_name in required_tools:

    tool_path = os.path.join(
        AMRFINDER_ENV,
        "bin",
        tool_name
    )

    if not os.path.exists(
        tool_path
    ):

        missing_tools.append(
            tool_name
        )


print()
print(
    f"Required companion tools available : "
    f"{len(missing_tools) == 0}"
)


if missing_tools:

    print(
        "Missing tools:",
        missing_tools
    )

else:

    print(
        "All required companion tools detected."
    )


# =========================================================
# 7. Final status
# =========================================================

if (
    os.path.exists(AMRFINDER_BIN)
    and len(missing_tools) == 0
):

    print()
    print(
        "PASS: AMRFinderPlus environment is ready."
    )

Installing micromamba...

Installing NCBI AMRFinderPlus...

AMRFINDERPLUS TOOL CHECK
4.2.7

Required companion tools available : True
All required companion tools detected.

PASS: AMRFinderPlus environment is ready.


## Cell 11 — Update and Verify AMRFinderPlus Database

In [ ]:
import os
import subprocess


# =========================================================
# 1. Update AMRFinderPlus database
# =========================================================

print("Updating AMRFinderPlus database...")

update_result = subprocess.run(
    [
        AMRFINDER_BIN,
        "--update"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(update_result.stdout)

if update_result.returncode != 0:
    raise RuntimeError(
        "AMRFinderPlus database update failed."
    )


# =========================================================
# 2. Check database version
# =========================================================

db_version_result = subprocess.run(
    [
        AMRFINDER_BIN,
        "--database_version"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True
)


database_version = (
    db_version_result.stdout
    .strip()
)


# =========================================================
# 3. Check software version again
# =========================================================

software_version_result = subprocess.run(
    [
        AMRFINDER_BIN,
        "--version"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True
)


software_version = (
    software_version_result.stdout
    .strip()
)


# =========================================================
# 4. Final status
# =========================================================

print()
print("=" * 70)
print("AMRFINDERPLUS DATABASE CHECK")
print("=" * 70)

print(
    f"Software version : "
    f"{software_version}"
)

print(
    f"Database version : "
    f"{database_version}"
)

print()

print(
    "PASS: AMRFinderPlus software and reference "
    "database are ready for genomic screening."
)

Updating AMRFinderPlus database...
Running: /content/amrfinder_env/bin/amrfinder --update
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /content/amrfinder_env/bin/
Software version: 4.2.7
Reverting to hard coded directory: /content/amrfinder_env/share/amrfinderplus/data/latest
Running: /content/amrfinder_env/bin/amrfinder_update -d /content/amrfinder_env/share/amrfinderplus/data
Looking up the published databases at https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance/AMRFinderPlus/database/
Looking for the target directory: /content/amrfinder_env/share/amrfinderplus/data/2026-08-07.1/
Running: /content/amrfinder_env/bin/amrfinder_index /content/amrfinder_env/share/amrfinderplus/data/2026-08-07.1/
Indexing
amrfinder_index took 5 seconds to complete
amrfinder_update took 41 seconds to complete
Database directory: /content/amrfinder_env/share/amrfinderplus/data/2026-08-07.1
Database version:

# Cell 12 — Pilot Known AMR Determinant Screening

In [ ]:
import os
import gzip
import shutil
import subprocess
import pandas as pd


# =========================================================
# 1. Pilot configuration
# =========================================================

PILOT_AMR_DIR = "/content/amrfinder_pilot"

os.makedirs(
    PILOT_AMR_DIR,
    exist_ok=True
)


# Balanced pilot: 3 Resistant + 3 Susceptible
pilot_r = (
    df_feature_input[
        df_feature_input[
            "Resistant Phenotype"
        ] == "Resistant"
    ]
    .head(3)
)

pilot_s = (
    df_feature_input[
        df_feature_input[
            "Resistant Phenotype"
        ] == "Susceptible"
    ]
    .head(3)
)

df_amr_pilot = (
    pd.concat(
        [
            pilot_r,
            pilot_s
        ],
        ignore_index=True
    )
)


print("Pilot cohort")
print("-" * 60)

display(
    df_amr_pilot[
        [
            "Genome ID",
            "Genome Name",
            "Resistant Phenotype",
            "FASTA Path"
        ]
    ]
)


# =========================================================
# 2. Confirm organism option is supported
# =========================================================

organism_result = subprocess.run(
    [
        AMRFINDER_BIN,
        "--list_organisms"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True
)


organism_text = (
    organism_result.stdout
)


organism_name = (
    "Klebsiella_pneumoniae"
)


if organism_name not in organism_text:

    raise RuntimeError(
        f"{organism_name} is not listed as a supported "
        "AMRFinderPlus organism option."
    )


print()
print(
    f"AMRFinderPlus organism option confirmed: "
    f"{organism_name}"
)


# =========================================================
# 3. Run AMRFinderPlus pilot
# =========================================================

pilot_results = []


for _, row in df_amr_pilot.iterrows():

    genome_id = str(
        row["Genome ID"]
    ).strip()

    phenotype = (
        row["Resistant Phenotype"]
    )

    fasta_gz_path = (
        row["FASTA Path"]
    )


    # -----------------------------------------------------
    # Decompress one genome to temporary FASTA
    # -----------------------------------------------------

    temp_fasta = os.path.join(
        PILOT_AMR_DIR,
        f"{genome_id}.fna"
    )

    output_tsv = os.path.join(
        PILOT_AMR_DIR,
        f"{genome_id}.amrfinder.tsv"
    )


    try:

        with gzip.open(
            fasta_gz_path,
            "rb"
        ) as src:

            with open(
                temp_fasta,
                "wb"
            ) as dst:

                shutil.copyfileobj(
                    src,
                    dst
                )


        # -------------------------------------------------
        # Run AMRFinderPlus
        # -------------------------------------------------

        command = [
            AMRFINDER_BIN,

            "--nucleotide",
            temp_fasta,

            "--organism",
            organism_name,

            "--threads",
            "2",

            "--name",
            genome_id,

            "--output",
            output_tsv
        ]


        run_result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )


        if run_result.returncode != 0:

            raise RuntimeError(
                run_result.stdout
            )


        # -------------------------------------------------
        # Read AMRFinderPlus output
        # -------------------------------------------------

        if (
            os.path.exists(output_tsv)
            and os.path.getsize(output_tsv) > 0
        ):

            df_hits = pd.read_csv(
                output_tsv,
                sep="\t"
            )

            hit_count = len(
                df_hits
            )

            if "Element symbol" in df_hits.columns:

                unique_elements = (
                    df_hits[
                        "Element symbol"
                    ]
                    .dropna()
                    .astype(str)
                    .nunique()
                )

            else:

                unique_elements = None

        else:

            hit_count = 0
            unique_elements = 0


        pilot_results.append({

            "Genome ID":
                genome_id,

            "Phenotype":
                phenotype,

            "AMRFinder Status":
                "PASS",

            "Total Hits":
                hit_count,

            "Unique Elements":
                unique_elements,

            "Output TSV":
                output_tsv,

            "Error":
                None
        })


    except Exception as e:

        pilot_results.append({

            "Genome ID":
                genome_id,

            "Phenotype":
                phenotype,

            "AMRFinder Status":
                "FAIL",

            "Total Hits":
                None,

            "Unique Elements":
                None,

            "Output TSV":
                None,

            "Error":
                str(e)
        })


    finally:

        # Remove temporary uncompressed FASTA
        if os.path.exists(
            temp_fasta
        ):

            os.remove(
                temp_fasta
            )


# =========================================================
# 4. Pilot summary
# =========================================================

df_amr_pilot_audit = pd.DataFrame(
    pilot_results
)


print()
print("=" * 70)
print("AMRFINDERPLUS PILOT SCREENING SUMMARY")
print("=" * 70)

display(
    df_amr_pilot_audit
)


successful_pilot = (
    df_amr_pilot_audit[
        "AMRFinder Status"
    ]
    .eq("PASS")
    .sum()
)


print()

print(
    f"Successful AMRFinderPlus runs : "
    f"{successful_pilot}/{len(df_amr_pilot_audit)}"
)


if successful_pilot == len(
    df_amr_pilot_audit
):

    print(
        "\nPASS: AMRFinderPlus screening succeeded "
        "for all pilot genomes."
    )

else:

    print(
        "\nWARNING: One or more pilot genomes "
        "failed AMRFinderPlus screening."
    )


# =========================================================
# 5. Inspect output columns from first successful genome
# =========================================================

successful_outputs = (
    df_amr_pilot_audit.loc[
        df_amr_pilot_audit[
            "AMRFinder Status"
        ] == "PASS",
        "Output TSV"
    ]
    .dropna()
    .tolist()
)


if successful_outputs:

    first_output = (
        successful_outputs[0]
    )

    df_first_amr_output = pd.read_csv(
        first_output,
        sep="\t"
    )

    print()
    print(
        "AMRFinderPlus output columns:"
    )

    print(
        df_first_amr_output.columns.tolist()
    )

    print()

    print(
        "First AMRFinderPlus hits:"
    )

    display(
        df_first_amr_output.head(10)
    )

Pilot cohort
------------------------------------------------------------


,Genome ID,Genome Name,Resistant Phenotype,FASTA Path
0,1284787.3,Klebsiella pneumoniae UHKPC40,Resistant,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
1,1284788.3,Klebsiella pneumoniae UHKPC23,Resistant,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
2,1284789.4,Klebsiella pneumoniae UHKPC28,Resistant,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
3,1284805.4,Klebsiella pneumoniae UHKPC17,Susceptible,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
4,1284814.3,Klebsiella pneumoniae UHKPC 52,Susceptible,/content/drive/MyDrive/AMR-Genome-ML/data/geno...
5,1284818.4,Klebsiella pneumoniae UHKPC179,Susceptible,/content/drive/MyDrive/AMR-Genome-ML/data/geno...



AMRFinderPlus organism option confirmed: Klebsiella_pneumoniae

AMRFINDERPLUS PILOT SCREENING SUMMARY


,Genome ID,Phenotype,AMRFinder Status,Total Hits,Unique Elements,Output TSV,Error
0,1284787.3,Resistant,PASS,17,17,/content/amrfinder_pilot/1284787.3.amrfinder.tsv,None
1,1284788.3,Resistant,PASS,18,18,/content/amrfinder_pilot/1284788.3.amrfinder.tsv,None
2,1284789.4,Resistant,PASS,12,12,/content/amrfinder_pilot/1284789.4.amrfinder.tsv,None
3,1284805.4,Susceptible,PASS,24,24,/content/amrfinder_pilot/1284805.4.amrfinder.tsv,None
4,1284814.3,Susceptible,PASS,15,15,/content/amrfinder_pilot/1284814.3.amrfinder.tsv,None
5,1284818.4,Susceptible,PASS,18,18,/content/amrfinder_pilot/1284818.4.amrfinder.tsv,None



Successful AMRFinderPlus runs : 6/6

PASS: AMRFinderPlus screening succeeded for all pilot genomes.

AMRFinderPlus output columns:
['Name', 'Protein id', 'Contig id', 'Start', 'Stop', 'Strand', 'Element symbol', 'Element name', 'Scope', 'Type', 'Subtype', 'Class', 'Subclass', 'Method', 'Target length', 'Reference sequence length', '% Coverage of reference', '% Identity to reference', 'Alignment length', 'Closest reference accession', 'Closest reference name', 'HMM accession', 'HMM description']

First AMRFinderPlus hits:


,Name,Protein id,Contig id,Start,Stop,Strand,Element symbol,Element name,Scope,Type,...,Method,Target length,Reference sequence length,% Coverage of reference,% Identity to reference,Alignment length,Closest reference accession,Closest reference name,HMM accession,HMM description
0,1284787.3,NaN,accn|AQOT01000033,7696,10326,-,gyrA_D87N,Klebsiella pneumoniae quinolone resistant GyrA,core,AMR,...,POINTX,877,877,100.0,99.66,877,WP_117036963.1,DNA gyrase subunit A GyrA,NaN,NaN
1,1284787.3,NaN,accn|AQOT01000033,7696,10326,-,gyrA_S83I,Klebsiella pneumoniae quinolone resistant GyrA,core,AMR,...,POINTX,877,877,100.0,99.66,877,WP_117036963.1,DNA gyrase subunit A GyrA,NaN,NaN
2,1284787.3,NaN,accn|AQOT01000051,70099,70515,-,fosA,FosA5 family fosfomycin resistance glutathione...,core,AMR,...,BLASTX,139,139,100.0,98.56,139,WP_004146118.1,FosA5 family fosfomycin resistance glutathione...,NaN,NaN
3,1284787.3,NaN,accn|AQOT01000052,416,1672,+,cmlA1,chloramphenicol efflux MFS transporter CmlA1,core,AMR,...,EXACTX,419,419,100.0,100.00,419,WP_000095725.1,chloramphenicol efflux MFS transporter CmlA1,NaN,NaN
4,1284787.3,NaN,accn|AQOT01000102,11971,13048,-,ompK35_E42RfsTer47,Klebsiella pneumoniae carbapenem resistant OmpK35,core,AMR,...,POINTX,359,359,100.0,100.00,359,WP_004141771.1,outer membrane porin OmpK35,NaN,NaN
5,1284787.3,NaN,accn|AQOT01000126,51,824,+,aac(3)-IVa,aminoglycoside N-acetyltransferase AAC(3)-IVa,core,AMR,...,EXACTX,258,258,100.0,100.00,258,WP_001199192.1,aminoglycoside N-acetyltransferase AAC(3)-IVa,NaN,NaN
6,1284787.3,NaN,accn|AQOT01000126,1056,2078,+,aph(4)-Ia,aminoglycoside O-phosphotransferase APH(4)-Ia,core,AMR,...,EXACTX,341,341,100.0,100.00,341,WP_000742814.1,aminoglycoside O-phosphotransferase APH(4)-Ia,NaN,NaN
7,1284787.3,NaN,accn|AQOT01000127,23,880,+,blaSHV-12,extended-spectrum class A beta-lactamase SHV-12,core,AMR,...,ALLELEX,286,286,100.0,100.00,286,WP_002904004.1,extended-spectrum class A beta-lactamase SHV-12,NaN,NaN
8,1284787.3,NaN,accn|AQOT01000136,163,492,+,qacL,quaternary ammonium compound efflux SMR transp...,core,STRESS,...,EXACTX,110,110,100.0,100.00,110,WP_000800531.1,quaternary ammonium compound efflux SMR transp...,NaN,NaN
9,1284787.3,NaN,accn|AQOT01000136,1678,2466,-,sul3,sulfonamide-resistant dihydropteroate synthase...,core,AMR,...,EXACTX,263,263,100.0,100.00,263,WP_000034420.1,sulfonamide-resistant dihydropteroate synthase...,NaN,NaN


In [ ]:
import os
import re
import shutil
import pandas as pd


# =========================================================
# Temporary checkpoint — persist Cell 12 pilot outputs
# =========================================================

PILOT_DRIVE_DIR = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_pilot"
)

os.makedirs(
    PILOT_DRIVE_DIR,
    exist_ok=True
)


# Copy all pilot TSV outputs
for filename in os.listdir(PILOT_AMR_DIR):

    source = os.path.join(
        PILOT_AMR_DIR,
        filename
    )

    destination = os.path.join(
        PILOT_DRIVE_DIR,
        filename
    )

    if os.path.isfile(source):
        shutil.copy2(
            source,
            destination
        )


# Save pilot audit
df_amr_pilot_audit.to_csv(
    os.path.join(
        PILOT_DRIVE_DIR,
        "amrfinder_pilot_audit.csv"
    ),
    index=False
)


# Save exact pilot cohort
df_amr_pilot.to_csv(
    os.path.join(
        PILOT_DRIVE_DIR,
        "amrfinder_pilot_cohort.csv"
    ),
    index=False
)


# Save reproducibility metadata
metadata = pd.DataFrame(
    [
        {
            "AMRFinderPlus Version": "4.2.7",
            "Database Version": "2026-08-07.1",
            "Organism": "Klebsiella_pneumoniae",
            "Pilot Genomes": len(df_amr_pilot),
            "Successful Runs": (
                df_amr_pilot_audit[
                    "AMRFinder Status"
                ]
                .eq("PASS")
                .sum()
            )
        }
    ]
)

metadata.to_csv(
    os.path.join(
        PILOT_DRIVE_DIR,
        "amrfinder_pilot_metadata.csv"
    ),
    index=False
)


print("Checkpoint saved to:")
print(PILOT_DRIVE_DIR)

print()
print("Files:")
for f in sorted(os.listdir(PILOT_DRIVE_DIR)):
    print(" -", f)

Checkpoint saved to:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/amrfinder_pilot

Files:
 - 1284787.3.amrfinder.tsv
 - 1284788.3.amrfinder.tsv
 - 1284789.4.amrfinder.tsv
 - 1284805.4.amrfinder.tsv
 - 1284814.3.amrfinder.tsv
 - 1284818.4.amrfinder.tsv
 - amrfinder_pilot_audit.csv
 - amrfinder_pilot_cohort.csv
 - amrfinder_pilot_metadata.csv


# Cell 13 — Resume from Drive and Audit Pilot AMRFinderPlus Outputs

In [ ]:
from google.colab import drive
import os
import glob
import pandas as pd


# =========================================================
# 1. Mount Google Drive
# =========================================================

drive.mount(
    "/content/drive"
)


# =========================================================
# 2. Persistent project paths
# =========================================================

DRIVE_ROOT = (
    "/content/drive/MyDrive/AMR-Genome-ML"
)

FEATURE_MANIFEST_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "processed",
    "genomic_feature_input_manifest.csv"
)

PILOT_CHECKPOINT_DIR = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_pilot"
)

PILOT_AUDIT_PATH = os.path.join(
    PILOT_CHECKPOINT_DIR,
    "amrfinder_pilot_audit.csv"
)

PILOT_COHORT_PATH = os.path.join(
    PILOT_CHECKPOINT_DIR,
    "amrfinder_pilot_cohort.csv"
)

PILOT_METADATA_PATH = os.path.join(
    PILOT_CHECKPOINT_DIR,
    "amrfinder_pilot_metadata.csv"
)

PILOT_COMBINED_HITS_PATH = os.path.join(
    PILOT_CHECKPOINT_DIR,
    "amrfinder_pilot_hits_combined.csv"
)


# =========================================================
# 3. Confirm checkpoint files exist
# =========================================================

required_files = [
    FEATURE_MANIFEST_PATH,
    PILOT_AUDIT_PATH,
    PILOT_COHORT_PATH,
    PILOT_METADATA_PATH
]


missing_files = [
    path
    for path in required_files
    if not os.path.exists(path)
]


if missing_files:

    raise FileNotFoundError(
        "Missing required persistent files:\n"
        + "\n".join(missing_files)
    )


# =========================================================
# 4. Reload persistent project tables
# =========================================================

df_feature_input = pd.read_csv(
    FEATURE_MANIFEST_PATH,
    dtype={
        "Genome ID": str,
        "Taxon ID": str
    }
)

df_amr_pilot_audit = pd.read_csv(
    PILOT_AUDIT_PATH,
    dtype={
        "Genome ID": str
    }
)

df_amr_pilot = pd.read_csv(
    PILOT_COHORT_PATH,
    dtype={
        "Genome ID": str,
        "Taxon ID": str
    }
)

df_amr_pilot_metadata = pd.read_csv(
    PILOT_METADATA_PATH
)


# =========================================================
# 5. Locate pilot AMRFinderPlus TSV files
# =========================================================

pilot_tsv_files = sorted(
    glob.glob(
        os.path.join(
            PILOT_CHECKPOINT_DIR,
            "*.amrfinder.tsv"
        )
    )
)


assert len(pilot_tsv_files) == 6, (
    f"Expected 6 pilot TSV files, "
    f"found {len(pilot_tsv_files)}."
)


# =========================================================
# 6. Load and combine pilot hits
# =========================================================

pilot_hit_tables = []


for tsv_path in pilot_tsv_files:

    df_hits = pd.read_csv(
        tsv_path,
        sep="\t"
    )

    genome_id_from_file = (
        os.path.basename(tsv_path)
        .replace(
            ".amrfinder.tsv",
            ""
        )
    )

    df_hits.insert(
        0,
        "Genome ID",
        genome_id_from_file
    )

    df_hits["Source File"] = (
        os.path.basename(tsv_path)
    )

    pilot_hit_tables.append(
        df_hits
    )


df_amr_pilot_hits = pd.concat(
    pilot_hit_tables,
    ignore_index=True
)


# =========================================================
# 7. Pilot integrity checks
# =========================================================

expected_pilot_ids = set(
    df_amr_pilot[
        "Genome ID"
    ]
    .astype(str)
)

observed_pilot_ids = set(
    df_amr_pilot_hits[
        "Genome ID"
    ]
    .astype(str)
)


assert (
    expected_pilot_ids
    == observed_pilot_ids
), (
    "Pilot Genome IDs in AMRFinder outputs "
    "do not match the saved pilot cohort."
)


assert (
    df_amr_pilot_audit[
        "AMRFinder Status"
    ]
    .eq("PASS")
    .all()
), (
    "One or more saved pilot runs "
    "were not successful."
)


assert (
    df_feature_input[
        "Genome ID"
    ]
    .nunique()
    == 4233
), (
    "Feature input manifest does not contain "
    "the expected 4,233 genomes."
)


# =========================================================
# 8. Save combined pilot hit table
# =========================================================

df_amr_pilot_hits.to_csv(
    PILOT_COMBINED_HITS_PATH,
    index=False
)


# =========================================================
# 9. Resume / schema audit summary
# =========================================================

print("=" * 70)
print("AMRFINDERPLUS PILOT CHECKPOINT RESTORED")
print("=" * 70)

print(
    f"Full feature manifest genomes : "
    f"{len(df_feature_input):,}"
)

print(
    f"Pilot genomes                 : "
    f"{len(df_amr_pilot):,}"
)

print(
    f"Pilot TSV files               : "
    f"{len(pilot_tsv_files):,}"
)

print(
    f"Combined AMRFinder hits       : "
    f"{len(df_amr_pilot_hits):,}"
)

print()

print("Stored reproducibility metadata:")
display(
    df_amr_pilot_metadata
)


# =========================================================
# 10. Output schema
# =========================================================

print()
print("AMRFinderPlus columns:")
print(
    df_amr_pilot_hits.columns.tolist()
)


# =========================================================
# 11. Determinant type audit
# =========================================================

print()
print("Type distribution:")
print(
    df_amr_pilot_hits[
        "Type"
    ]
    .value_counts(
        dropna=False
    )
)


print()
print("Subtype distribution:")
print(
    df_amr_pilot_hits[
        "Subtype"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
)


# =========================================================
# 12. AMR class audit
# =========================================================

print()
print("Class distribution:")
print(
    df_amr_pilot_hits[
        "Class"
    ]
    .value_counts(
        dropna=False
    )
    .head(20)
)


# =========================================================
# 13. Detection-method audit
# =========================================================

print()
print("Detection method distribution:")
print(
    df_amr_pilot_hits[
        "Method"
    ]
    .value_counts(
        dropna=False
    )
)


# =========================================================
# 14. Example determinants
# =========================================================

preview_columns = [
    "Genome ID",
    "Element symbol",
    "Element name",
    "Scope",
    "Type",
    "Subtype",
    "Class",
    "Subclass",
    "Method"
]


preview_columns = [
    col
    for col in preview_columns
    if col in df_amr_pilot_hits.columns
]


print()
print("Example pilot determinants:")

display(
    df_amr_pilot_hits[
        preview_columns
    ]
    .head(20)
)


print()
print(
    "Combined pilot hit table saved to:"
)

print(
    PILOT_COMBINED_HITS_PATH
)

print()

print(
    "PASS: Cell 13 restored the project state "
    "from Google Drive without rerunning Cells 1–12."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AMRFINDERPLUS PILOT CHECKPOINT RESTORED
Full feature manifest genomes : 4,233
Pilot genomes                 : 6
Pilot TSV files               : 6
Combined AMRFinder hits       : 104

Stored reproducibility metadata:


,AMRFinderPlus Version,Database Version,Organism,Pilot Genomes,Successful Runs
0,4.2.7,2026-08-07.1,Klebsiella_pneumoniae,6,6



AMRFinderPlus columns:
['Genome ID', 'Name', 'Protein id', 'Contig id', 'Start', 'Stop', 'Strand', 'Element symbol', 'Element name', 'Scope', 'Type', 'Subtype', 'Class', 'Subclass', 'Method', 'Target length', 'Reference sequence length', '% Coverage of reference', '% Identity to reference', 'Alignment length', 'Closest reference accession', 'Closest reference name', 'HMM accession', 'HMM description', 'Source File']

Type distribution:
Type
AMR       99
STRESS     5
Name: count, dtype: int64

Subtype distribution:
Subtype
AMR              73
POINT            21
POINT_DISRUPT     5
BIOCIDE           5
Name: count, dtype: int64

Class distribution:
Class
BETA-LACTAM                                   24
AMINOGLYCOSIDE                                16
QUINOLONE                                     15
NITROFURAN/PHENICOL/QUINOLONE/TETRACYCLINE    12
MACROLIDE                                      8
FOSFOMYCIN                                     6
SULFONAMIDE                                 

,Genome ID,Element symbol,Element name,Scope,Type,Subtype,Class,Subclass,Method
0,1284787.3,gyrA_D87N,Klebsiella pneumoniae quinolone resistant GyrA,core,AMR,POINT,QUINOLONE,QUINOLONE,POINTX
1,1284787.3,gyrA_S83I,Klebsiella pneumoniae quinolone resistant GyrA,core,AMR,POINT,QUINOLONE,QUINOLONE,POINTX
2,1284787.3,fosA,FosA5 family fosfomycin resistance glutathione...,core,AMR,AMR,FOSFOMYCIN,FOSFOMYCIN,BLASTX
3,1284787.3,cmlA1,chloramphenicol efflux MFS transporter CmlA1,core,AMR,AMR,PHENICOL,CHLORAMPHENICOL,EXACTX
4,1284787.3,ompK35_E42RfsTer47,Klebsiella pneumoniae carbapenem resistant OmpK35,core,AMR,POINT_DISRUPT,BETA-LACTAM,CARBAPENEM,POINTX
5,1284787.3,aac(3)-IVa,aminoglycoside N-acetyltransferase AAC(3)-IVa,core,AMR,AMR,AMINOGLYCOSIDE,APRAMYCIN/GENTAMICIN/TOBRAMYCIN,EXACTX
6,1284787.3,aph(4)-Ia,aminoglycoside O-phosphotransferase APH(4)-Ia,core,AMR,AMR,AMINOGLYCOSIDE,HYGROMYCIN,EXACTX
7,1284787.3,blaSHV-12,extended-spectrum class A beta-lactamase SHV-12,core,AMR,AMR,BETA-LACTAM,CEFIDEROCOL/CEPHALOSPORIN,ALLELEX
8,1284787.3,qacL,quaternary ammonium compound efflux SMR transp...,core,STRESS,BIOCIDE,QUATERNARY AMMONIUM,QUATERNARY AMMONIUM,EXACTX
9,1284787.3,sul3,sulfonamide-resistant dihydropteroate synthase...,core,AMR,AMR,SULFONAMIDE,SULFONAMIDE,EXACTX



Combined pilot hit table saved to:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/amrfinder_pilot/amrfinder_pilot_hits_combined.csv

PASS: Cell 13 restored the project state from Google Drive without rerunning Cells 1–12.


# Cell 14 — Define Known AMR Determinant Inclusion Rules

In [ ]:
import os
import pandas as pd


# =========================================================
# 1. Define phenotype-independent inclusion policy
# =========================================================

PRIMARY_INCLUDED_TYPES = {
    "AMR"
}

PRIMARY_INCLUDED_SUBTYPES = {
    "AMR",
    "POINT",
    "POINT_DISRUPT"
}

EXCLUDED_TYPES = {
    "STRESS"
}


# =========================================================
# 2. Apply rule to pilot hits
# =========================================================

df_amr_pilot_hits = (
    df_amr_pilot_hits
    .copy()
)


df_amr_pilot_hits["Include in Known AMR"] = (
    df_amr_pilot_hits["Type"]
    .isin(PRIMARY_INCLUDED_TYPES)
    &
    df_amr_pilot_hits["Subtype"]
    .isin(PRIMARY_INCLUDED_SUBTYPES)
)


df_known_amr_pilot = (
    df_amr_pilot_hits[
        df_amr_pilot_hits[
            "Include in Known AMR"
        ]
    ]
    .copy()
)


df_excluded_pilot = (
    df_amr_pilot_hits[
        ~df_amr_pilot_hits[
            "Include in Known AMR"
        ]
    ]
    .copy()
)


# =========================================================
# 3. Construct canonical determinant feature key
# =========================================================

df_known_amr_pilot[
    "Determinant Feature"
] = (
    df_known_amr_pilot[
        "Element symbol"
    ]
    .astype(str)
    .str.strip()
)


# =========================================================
# 4. Integrity checks
# =========================================================

assert (
    df_known_amr_pilot[
        "Type"
    ]
    .eq("AMR")
    .all()
), (
    "Non-AMR entries remain in the primary "
    "known-AMR representation."
)


assert (
    df_known_amr_pilot[
        "Determinant Feature"
    ]
    .notna()
    .all()
), (
    "Missing determinant feature identifiers detected."
)


assert (
    ~df_known_amr_pilot[
        "Determinant Feature"
    ]
    .eq("")
    .any()
), (
    "Empty determinant feature identifiers detected."
)


# =========================================================
# 5. Feature-level pilot summary
# =========================================================

pilot_feature_prevalence = (
    df_known_amr_pilot
    .groupby(
        "Determinant Feature"
    )["Genome ID"]
    .nunique()
    .sort_values(
        ascending=False
    )
    .rename(
        "Genome Count"
    )
    .reset_index()
)


# =========================================================
# 6. Per-genome determinant counts
# =========================================================

pilot_genome_counts = (
    df_known_amr_pilot
    .groupby(
        "Genome ID"
    )["Determinant Feature"]
    .nunique()
    .rename(
        "Known AMR Determinants"
    )
    .reset_index()
)


pilot_genome_counts = (
    df_amr_pilot[
        [
            "Genome ID",
            "Resistant Phenotype"
        ]
    ]
    .merge(
        pilot_genome_counts,
        on="Genome ID",
        how="left"
    )
)


pilot_genome_counts[
    "Known AMR Determinants"
] = (
    pilot_genome_counts[
        "Known AMR Determinants"
    ]
    .fillna(0)
    .astype(int)
)


# =========================================================
# 7. Save inclusion-policy audit
# =========================================================

POLICY_OUTPUT_DIR = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_pilot"
)


df_known_amr_pilot.to_csv(
    os.path.join(
        POLICY_OUTPUT_DIR,
        "amrfinder_pilot_known_amr_included.csv"
    ),
    index=False
)


df_excluded_pilot.to_csv(
    os.path.join(
        POLICY_OUTPUT_DIR,
        "amrfinder_pilot_known_amr_excluded.csv"
    ),
    index=False
)


pilot_feature_prevalence.to_csv(
    os.path.join(
        POLICY_OUTPUT_DIR,
        "amrfinder_pilot_feature_prevalence.csv"
    ),
    index=False
)


# =========================================================
# 8. Summary
# =========================================================

print("=" * 70)
print("KNOWN AMR DETERMINANT INCLUSION POLICY")
print("=" * 70)

print(
    f"Total pilot hits             : "
    f"{len(df_amr_pilot_hits):,}"
)

print(
    f"Included AMR hits            : "
    f"{len(df_known_amr_pilot):,}"
)

print(
    f"Excluded non-AMR hits        : "
    f"{len(df_excluded_pilot):,}"
)

print(
    f"Unique determinant features  : "
    f"{df_known_amr_pilot['Determinant Feature'].nunique():,}"
)

print()

print("Included subtype distribution:")
print(
    df_known_amr_pilot[
        "Subtype"
    ]
    .value_counts()
)

print()

print("Excluded type/subtype combinations:")
print(
    df_excluded_pilot[
        [
            "Type",
            "Subtype"
        ]
    ]
    .value_counts()
)

print()

print("Known AMR determinants per pilot genome:")
display(
    pilot_genome_counts
)

print()

print("Most prevalent pilot determinant features:")
display(
    pilot_feature_prevalence.head(20)
)

print()

print(
    "PASS: Phenotype-independent known-AMR "
    "determinant inclusion rules are fixed."
)

KNOWN AMR DETERMINANT INCLUSION POLICY
Total pilot hits             : 104
Included AMR hits            : 99
Excluded non-AMR hits        : 5
Unique determinant features  : 42

Included subtype distribution:
Subtype
AMR              73
POINT            21
POINT_DISRUPT     5
Name: count, dtype: int64

Excluded type/subtype combinations:
Type    Subtype
STRESS  BIOCIDE    5
Name: count, dtype: int64

Known AMR determinants per pilot genome:


,Genome ID,Resistant Phenotype,Known AMR Determinants
0,1284787.3,Resistant,16
1,1284788.3,Resistant,17
2,1284789.4,Resistant,12
3,1284805.4,Susceptible,22
4,1284814.3,Susceptible,14
5,1284818.4,Susceptible,18



Most prevalent pilot determinant features:


,Determinant Feature,Genome Count
0,oqxA,6
1,fosA,6
2,oqxB,6
3,parC_S80I,6
4,blaSHV-12,5
5,gyrA_S83I,5
6,ompK35_E42RfsTer47,5
7,aadA2,4
8,dfrA12,3
9,aac(6')-Ib,3



PASS: Phenotype-independent known-AMR determinant inclusion rules are fixed.


# Cell 15 — Restore and Verify the AMRFinderPlus Runtime

In [ ]:
import os
import re
import subprocess
import pandas as pd


# =========================================================
# 1. Expected versions from the completed pilot
# =========================================================

EXPECTED_SOFTWARE_VERSION = "4.2.7"
EXPECTED_DATABASE_VERSION = "2026-08-07.1"


# =========================================================
# 2. Runtime paths
# =========================================================

MICROMAMBA_ROOT = "/content/micromamba"

MICROMAMBA_BIN = os.path.join(
    MICROMAMBA_ROOT,
    "bin",
    "micromamba"
)

AMRFINDER_ENV = "/content/amrfinder_env"

AMRFINDER_BIN = os.path.join(
    AMRFINDER_ENV,
    "bin",
    "amrfinder"
)


# =========================================================
# 3. Install micromamba if absent
# =========================================================

if not os.path.exists(MICROMAMBA_BIN):

    print("Installing micromamba...")

    os.makedirs(
        MICROMAMBA_ROOT,
        exist_ok=True
    )

    install_micromamba = f"""
    set -e

    curl -Ls \
        https://micro.mamba.pm/api/micromamba/linux-64/latest \
        | tar -xj \
        -C {MICROMAMBA_ROOT} \
        bin/micromamba
    """

    subprocess.run(
        install_micromamba,
        shell=True,
        check=True,
        executable="/bin/bash"
    )

else:

    print(
        "micromamba already available."
    )


# =========================================================
# 4. Install exact AMRFinderPlus software version
# =========================================================

if not os.path.exists(AMRFINDER_BIN):

    print()
    print(
        "Installing AMRFinderPlus "
        f"{EXPECTED_SOFTWARE_VERSION}..."
    )

    subprocess.run(
        [
            MICROMAMBA_BIN,
            "create",
            "-y",
            "-p",
            AMRFINDER_ENV,

            "-c",
            "conda-forge",

            "-c",
            "bioconda",

            (
                "ncbi-amrfinderplus="
                f"{EXPECTED_SOFTWARE_VERSION}"
            )
        ],
        check=True
    )

else:

    print()
    print(
        "AMRFinderPlus environment already exists."
    )


assert os.path.exists(
    AMRFINDER_BIN
), "AMRFinderPlus executable was not found."


# =========================================================
# 5. Runtime environment variables
# =========================================================

amr_runtime_env = os.environ.copy()

amr_runtime_env[
    "CONDA_PREFIX"
] = AMRFINDER_ENV

amr_runtime_env[
    "PATH"
] = (
    os.path.join(
        AMRFINDER_ENV,
        "bin"
    )
    + os.pathsep
    + amr_runtime_env.get(
        "PATH",
        ""
    )
)


# =========================================================
# 6. Verify software version
# =========================================================

software_result = subprocess.run(
    [
        AMRFINDER_BIN,
        "--version"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True,
    env=amr_runtime_env
)


software_text = (
    software_result.stdout
    .strip()
)


software_match = re.search(
    r"\b(\d+\.\d+\.\d+)\b",
    software_text
)


if software_match is None:

    raise RuntimeError(
        "Could not parse AMRFinderPlus "
        "software version."
    )


software_version = (
    software_match.group(1)
)


if (
    software_version
    != EXPECTED_SOFTWARE_VERSION
):

    raise RuntimeError(
        "AMRFinderPlus software version mismatch.\n"
        f"Expected: {EXPECTED_SOFTWARE_VERSION}\n"
        f"Observed: {software_version}"
    )


# =========================================================
# 7. Restore / update reference database
# =========================================================

print()
print(
    "Restoring AMRFinderPlus database..."
)


update_result = subprocess.run(
    [
        AMRFINDER_BIN,
        "--update"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=amr_runtime_env
)


print(
    update_result.stdout
)


if update_result.returncode != 0:

    raise RuntimeError(
        "AMRFinderPlus database setup failed."
    )


# =========================================================
# 8. Read exact database version
# =========================================================

db_result = subprocess.run(
    [
        AMRFINDER_BIN,
        "--database_version"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=True,
    env=amr_runtime_env
)


db_text = (
    db_result.stdout
)


db_match = re.search(
    r"Database version:\s*([^\s]+)",
    db_text
)


if db_match is None:

    raise RuntimeError(
        "Could not parse AMRFinderPlus "
        "database version."
    )


database_version = (
    db_match.group(1)
)


# =========================================================
# 9. Enforce database reproducibility
# =========================================================

if (
    database_version
    != EXPECTED_DATABASE_VERSION
):

    raise RuntimeError(
        "AMRFinderPlus database version mismatch.\n"
        f"Pilot database : "
        f"{EXPECTED_DATABASE_VERSION}\n"
        f"Current database: "
        f"{database_version}\n\n"
        "Do not begin full-cohort screening "
        "with a different database version."
    )


# =========================================================
# 10. Verify required companion tools
# =========================================================

required_tools = [
    "blastn",
    "blastp",
    "hmmsearch"
]


missing_tools = []


for tool_name in required_tools:

    tool_path = os.path.join(
        AMRFINDER_ENV,
        "bin",
        tool_name
    )

    if not os.path.exists(
        tool_path
    ):

        missing_tools.append(
            tool_name
        )


if missing_tools:

    raise RuntimeError(
        "Missing required AMRFinderPlus "
        f"dependencies: {missing_tools}"
    )


# =========================================================
# 11. Save runtime verification record
# =========================================================

RUNTIME_METADATA_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_runtime_verified.csv"
)


df_runtime_metadata = pd.DataFrame(
    [
        {
            "AMRFinderPlus Version":
                software_version,

            "Database Version":
                database_version,

            "Organism":
                "Klebsiella_pneumoniae",

            "Software Match":
                (
                    software_version
                    == EXPECTED_SOFTWARE_VERSION
                ),

            "Database Match":
                (
                    database_version
                    == EXPECTED_DATABASE_VERSION
                )
        }
    ]
)


df_runtime_metadata.to_csv(
    RUNTIME_METADATA_PATH,
    index=False
)


# =========================================================
# 12. Final status
# =========================================================

print()
print("=" * 70)
print("AMRFINDERPLUS RUNTIME VERIFICATION")
print("=" * 70)

print(
    f"Software version : "
    f"{software_version}"
)

print(
    f"Database version : "
    f"{database_version}"
)

print(
    "Organism         : "
    "Klebsiella_pneumoniae"
)

print(
    "Companion tools  : "
    "PASS"
)

print()

print(
    "PASS: AMRFinderPlus runtime exactly matches "
    "the pilot screening environment."
)

print()

print(
    "Runtime verification saved to:"
)

print(
    RUNTIME_METADATA_PATH
)

micromamba already available.

AMRFinderPlus environment already exists.

Restoring AMRFinderPlus database...
Running: /content/amrfinder_env/bin/amrfinder --update
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /content/amrfinder_env/bin/
Software version: 4.2.7
Running: /content/amrfinder_env/bin/amrfinder_update -d /content/amrfinder_env/share/amrfinderplus/data
Looking up the published databases at https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance/AMRFinderPlus/database/
Looking for the target directory: /content/amrfinder_env/share/amrfinderplus/data/2026-08-07.1/
Skipping update
Use amrfinder --force_update to overwrite the existing database
amrfinder_update took 1 seconds to complete
Database directory: /content/amrfinder_env/share/amrfinderplus/data/2026-08-07.1
Database version: 2026-08-07.1
amrfinder took 1 seconds to complete


AMRFINDERPLUS RUNTIME VERIFICATION
Software versi

# Cell 16 — Configure Persistent Full-Cohort AMRFinderPlus Screening

In [ ]:
import os
import pandas as pd


# =========================================================
# 1. Persistent output locations
# =========================================================

KNOWN_AMR_ROOT = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr"
)

FULL_AMR_RAW_DIR = os.path.join(
    KNOWN_AMR_ROOT,
    "raw_amrfinder"
)

FULL_AMR_JOB_MANIFEST_PATH = os.path.join(
    KNOWN_AMR_ROOT,
    "amrfinder_full_job_manifest.csv"
)

FULL_AMR_AUDIT_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_full_screening_audit.csv"
)


# Local temporary workspace
FULL_AMR_TEMP_DIR = (
    "/content/amrfinder_full_temp"
)


for directory in [
    KNOWN_AMR_ROOT,
    FULL_AMR_RAW_DIR,
    FULL_AMR_TEMP_DIR
]:

    os.makedirs(
        directory,
        exist_ok=True
    )


# =========================================================
# 2. Verify full feature input cohort
# =========================================================

assert len(df_feature_input) == 4233, (
    f"Expected 4,233 genomes, "
    f"found {len(df_feature_input):,}."
)


assert (
    df_feature_input[
        "Genome ID"
    ]
    .nunique()
    == 4233
), (
    "Duplicate Genome IDs detected."
)


assert (
    df_feature_input[
        "Complete FASTA"
    ]
    .eq(True)
    .all()
), (
    "One or more genomes lack a validated FASTA."
)


# =========================================================
# 3. Build phenotype-independent screening manifest
# =========================================================

df_amr_jobs = (
    df_feature_input[
        [
            "Genome ID",
            "FASTA Path"
        ]
    ]
    .copy()
)


df_amr_jobs[
    "Genome ID"
] = (
    df_amr_jobs[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


df_amr_jobs[
    "AMRFinder Output"
] = (
    df_amr_jobs[
        "Genome ID"
    ]
    .apply(
        lambda genome_id:
            os.path.join(
                FULL_AMR_RAW_DIR,
                f"{genome_id}.amrfinder.tsv"
            )
    )
)


# =========================================================
# 4. Job-manifest integrity checks
# =========================================================

assert (
    df_amr_jobs[
        "Genome ID"
    ]
    .nunique()
    == len(df_amr_jobs)
), (
    "Genome IDs are not unique in "
    "the screening manifest."
)


assert (
    df_amr_jobs[
        "FASTA Path"
    ]
    .nunique()
    == len(df_amr_jobs)
), (
    "FASTA paths are not unique."
)


assert (
    df_amr_jobs[
        "AMRFinder Output"
    ]
    .nunique()
    == len(df_amr_jobs)
), (
    "AMRFinder output paths are not unique."
)


# =========================================================
# 5. Confirm representative FASTA accessibility
# =========================================================

representative_rows = pd.concat(
    [
        df_amr_jobs.head(3),
        df_amr_jobs.tail(3)
    ],
    ignore_index=True
)


representative_missing = (
    representative_rows.loc[
        ~representative_rows[
            "FASTA Path"
        ]
        .apply(os.path.exists),
        "FASTA Path"
    ]
    .tolist()
)


if representative_missing:

    raise FileNotFoundError(
        "Representative FASTA accessibility "
        "check failed:\n"
        + "\n".join(
            representative_missing
        )
    )


# =========================================================
# 6. Detect previously completed outputs
#    (supports future resume)
# =========================================================

df_amr_jobs[
    "Output Exists"
] = (
    df_amr_jobs[
        "AMRFinder Output"
    ]
    .apply(os.path.exists)
)


existing_outputs = int(
    df_amr_jobs[
        "Output Exists"
    ]
    .sum()
)


remaining_jobs = (
    len(df_amr_jobs)
    - existing_outputs
)


# =========================================================
# 7. Save persistent job manifest
# =========================================================

df_amr_jobs[
    [
        "Genome ID",
        "FASTA Path",
        "AMRFinder Output"
    ]
].to_csv(
    FULL_AMR_JOB_MANIFEST_PATH,
    index=False
)


# =========================================================
# 8. Screening configuration
# =========================================================

AMRFINDER_ORGANISM = (
    "Klebsiella_pneumoniae"
)

AMRFINDER_THREADS_PER_JOB = 2

AMRFINDER_SOFTWARE_VERSION = (
    "4.2.7"
)

AMRFINDER_DATABASE_VERSION = (
    "2026-08-07.1"
)


# =========================================================
# 9. Final preflight summary
# =========================================================

print("=" * 70)
print("FULL-COHORT AMRFINDERPLUS SCREENING CONFIGURATION")
print("=" * 70)

print(
    f"Total genomes              : "
    f"{len(df_amr_jobs):,}"
)

print(
    f"Unique Genome IDs          : "
    f"{df_amr_jobs['Genome ID'].nunique():,}"
)

print(
    f"Existing outputs           : "
    f"{existing_outputs:,}"
)

print(
    f"Remaining genomes          : "
    f"{remaining_jobs:,}"
)

print()

print(
    f"AMRFinderPlus version      : "
    f"{AMRFINDER_SOFTWARE_VERSION}"
)

print(
    f"Database version           : "
    f"{AMRFINDER_DATABASE_VERSION}"
)

print(
    f"Organism                   : "
    f"{AMRFINDER_ORGANISM}"
)

print(
    f"Threads per genome         : "
    f"{AMRFINDER_THREADS_PER_JOB}"
)

print()

print(
    "Phenotype used in screening: NO"
)

print()

print(
    "Persistent raw output directory:"
)

print(
    FULL_AMR_RAW_DIR
)

print()

print(
    "Job manifest saved to:"
)

print(
    FULL_AMR_JOB_MANIFEST_PATH
)

print()

print(
    "PASS: Full-cohort AMRFinderPlus screening "
    "is configured and ready to run."
)

FULL-COHORT AMRFINDERPLUS SCREENING CONFIGURATION
Total genomes              : 4,233
Unique Genome IDs          : 4,233
Existing outputs           : 8
Remaining genomes          : 4,225

AMRFinderPlus version      : 4.2.7
Database version           : 2026-08-07.1
Organism                   : Klebsiella_pneumoniae
Threads per genome         : 2

Phenotype used in screening: NO

Persistent raw output directory:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/raw_amrfinder

Job manifest saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/known_amr/amrfinder_full_job_manifest.csv

PASS: Full-cohort AMRFinderPlus screening is configured and ready to run.


# Cell 17 — Benchmark Local vs Google Drive AMRFinderPlus I/O

In [ ]:
import os
import gzip
import shutil
import subprocess
import tempfile
import time
import pandas as pd


# =========================================================
# 1. Select six unfinished genomes
# =========================================================

existing_outputs = {
    f.replace(".amrfinder.tsv", "")
    for f in os.listdir(FULL_AMR_RAW_DIR)
    if f.endswith(".amrfinder.tsv")
}


df_benchmark = (
    df_amr_jobs[
        ~df_amr_jobs["Genome ID"]
        .astype(str)
        .isin(existing_outputs)
    ]
    .head(6)
    .copy()
    .reset_index(drop=True)
)


print(
    f"Benchmark genomes: "
    f"{len(df_benchmark)}"
)


# =========================================================
# 2. Local benchmark workspace
# =========================================================

BENCHMARK_ROOT = "/content/amrfinder_benchmark"

LOCAL_GZ_DIR = os.path.join(
    BENCHMARK_ROOT,
    "fasta_gz"
)

LOCAL_RESULT_DIR = os.path.join(
    BENCHMARK_ROOT,
    "results"
)


os.makedirs(
    LOCAL_GZ_DIR,
    exist_ok=True
)

os.makedirs(
    LOCAL_RESULT_DIR,
    exist_ok=True
)


# =========================================================
# 3. Copy compressed FASTA files from Drive to local disk
# =========================================================

copy_records = []


for _, row in df_benchmark.iterrows():

    genome_id = str(
        row["Genome ID"]
    )

    source = row[
        "FASTA Path"
    ]

    destination = os.path.join(
        LOCAL_GZ_DIR,
        f"{genome_id}.fna.gz"
    )


    start = time.time()

    shutil.copy2(
        source,
        destination
    )

    copy_seconds = (
        time.time()
        - start
    )


    copy_records.append(
        {
            "Genome ID":
                genome_id,

            "Copy Seconds":
                copy_seconds,

            "Local FASTA":
                destination
        }
    )


df_copy_benchmark = pd.DataFrame(
    copy_records
)


# =========================================================
# 4. Run AMRFinderPlus sequentially from local storage
# =========================================================

run_records = []


for _, row in df_copy_benchmark.iterrows():

    genome_id = row[
        "Genome ID"
    ]

    fasta_gz = row[
        "Local FASTA"
    ]

    temp_fasta = os.path.join(
        BENCHMARK_ROOT,
        f"{genome_id}.fna"
    )

    output_tsv = os.path.join(
        LOCAL_RESULT_DIR,
        f"{genome_id}.amrfinder.tsv"
    )


    # -----------------------------------------------------
    # Local decompression
    # -----------------------------------------------------

    start_decompress = time.time()

    with gzip.open(
        fasta_gz,
        "rb"
    ) as src:

        with open(
            temp_fasta,
            "wb"
        ) as dst:

            shutil.copyfileobj(
                src,
                dst
            )


    decompress_seconds = (
        time.time()
        - start_decompress
    )


    # -----------------------------------------------------
    # AMRFinderPlus
    # -----------------------------------------------------

    start_amr = time.time()

    result = subprocess.run(
        [
            AMRFINDER_BIN,

            "--nucleotide",
            temp_fasta,

            "--organism",
            AMRFINDER_ORGANISM,

            "--threads",
            "2",

            "--name",
            genome_id,

            "--output",
            output_tsv
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=amr_runtime_env
    )


    amr_seconds = (
        time.time()
        - start_amr
    )


    os.remove(
        temp_fasta
    )


    if result.returncode != 0:

        raise RuntimeError(
            result.stdout
        )


    df_result = pd.read_csv(
        output_tsv,
        sep="\t"
    )


    run_records.append(
        {
            "Genome ID":
                genome_id,

            "Copy Seconds":
                float(
                    df_copy_benchmark.loc[
                        df_copy_benchmark[
                            "Genome ID"
                        ] == genome_id,
                        "Copy Seconds"
                    ].iloc[0]
                ),

            "Decompress Seconds":
                decompress_seconds,

            "AMRFinder Seconds":
                amr_seconds,

            "Hits":
                len(df_result)
        }
    )


# =========================================================
# 5. Benchmark summary
# =========================================================

df_benchmark_results = pd.DataFrame(
    run_records
)


print()
print("=" * 70)
print("AMRFINDERPLUS LOCAL-I/O BENCHMARK")
print("=" * 70)

display(
    df_benchmark_results
)


print()

print(
    f"Mean Drive → local copy   : "
    f"{df_benchmark_results['Copy Seconds'].mean():.2f} s/genome"
)

print(
    f"Mean local decompression  : "
    f"{df_benchmark_results['Decompress Seconds'].mean():.2f} s/genome"
)

print(
    f"Mean AMRFinder compute    : "
    f"{df_benchmark_results['AMRFinder Seconds'].mean():.2f} s/genome"
)

print()

print(
    "Total mean after FASTA is local: "
    f"{(
        df_benchmark_results['Decompress Seconds']
        + df_benchmark_results['AMRFinder Seconds']
    ).mean():.2f} s/genome"
)

Benchmark genomes: 6

AMRFINDERPLUS LOCAL-I/O BENCHMARK


,Genome ID,Copy Seconds,Decompress Seconds,AMRFinder Seconds,Hits
0,1284801.3,0.009061,0.035275,35.171167,24
1,1284802.4,0.007338,0.036176,36.808480,17
2,1284803.4,0.626413,0.034485,35.276273,24
3,1284804.4,0.960844,0.033382,34.033841,19
4,1284805.4,0.483259,0.051122,36.055457,24
5,1284806.4,0.672520,0.037884,33.851716,21



Mean Drive → local copy   : 0.46 s/genome
Mean local decompression  : 0.04 s/genome
Mean AMRFinder compute    : 35.20 s/genome

Total mean after FASTA is local: 35.24 s/genome


# Cell 18 — Benchmark Prodigal-Assisted Combined AMRFinderPlus Mode

In [ ]:
import os
import gzip
import shutil
import subprocess
import tempfile
import time
import pandas as pd


# =========================================================
# 1. Install / verify Prodigal
# =========================================================

PRODIGAL_BIN = os.path.join(
    AMRFINDER_ENV,
    "bin",
    "prodigal"
)


if not os.path.exists(PRODIGAL_BIN):

    print("Installing Prodigal 2.6.3...")

    subprocess.run(
        [
            MICROMAMBA_BIN,
            "install",
            "-y",
            "-p",
            AMRFINDER_ENV,
            "-c",
            "conda-forge",
            "-c",
            "bioconda",
            "prodigal=2.6.3"
        ],
        check=True
    )


assert os.path.exists(PRODIGAL_BIN)


prodigal_version_result = subprocess.run(
    [
        PRODIGAL_BIN,
        "-v"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


print(
    prodigal_version_result.stdout.strip()
)


# =========================================================
# 2. Benchmark output location
# =========================================================

COMBINED_BENCHMARK_DIR = (
    "/content/amrfinder_combined_benchmark"
)

os.makedirs(
    COMBINED_BENCHMARK_DIR,
    exist_ok=True
)


COMBINED_BENCHMARK_RESULT_PATH = os.path.join(
    PILOT_CHECKPOINT_DIR,
    "amrfinder_combined_mode_benchmark.csv"
)


# =========================================================
# 3. Use the same six original pilot genomes
# =========================================================

df_combined_benchmark = (
    df_amr_pilot[
        [
            "Genome ID",
            "FASTA Path"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(df_combined_benchmark) == 6


# =========================================================
# 4. Run Prodigal + AMRFinder combined mode
# =========================================================

benchmark_records = []


for _, row in df_combined_benchmark.iterrows():

    genome_id = str(
        row["Genome ID"]
    )

    fasta_gz_path = (
        row["FASTA Path"]
    )


    temp_dir = tempfile.mkdtemp(
        prefix=f"{genome_id}_",
        dir=COMBINED_BENCHMARK_DIR
    )


    nucleotide_fasta = os.path.join(
        temp_dir,
        f"{genome_id}.fna"
    )

    protein_fasta = os.path.join(
        temp_dir,
        f"{genome_id}.faa"
    )

    prodigal_gff = os.path.join(
        temp_dir,
        f"{genome_id}.gff"
    )

    combined_output = os.path.join(
        temp_dir,
        f"{genome_id}.combined.tsv"
    )


    try:

        # -------------------------------------------------
        # Decompress FASTA
        # -------------------------------------------------

        with gzip.open(
            fasta_gz_path,
            "rb"
        ) as src:

            with open(
                nucleotide_fasta,
                "wb"
            ) as dst:

                shutil.copyfileobj(
                    src,
                    dst
                )


        # -------------------------------------------------
        # Prodigal gene prediction
        # -------------------------------------------------

        prodigal_start = time.time()


        prodigal_result = subprocess.run(
            [
                PRODIGAL_BIN,
                "-i",
                nucleotide_fasta,

                "-a",
                protein_fasta,

                "-o",
                prodigal_gff,

                "-f",
                "gff",

                "-p",
                "single",

                "-q"
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=amr_runtime_env
        )


        prodigal_seconds = (
            time.time()
            - prodigal_start
        )


        if prodigal_result.returncode != 0:

            raise RuntimeError(
                "Prodigal failed:\n"
                + prodigal_result.stdout
            )


        assert os.path.exists(
            protein_fasta
        )

        assert os.path.exists(
            prodigal_gff
        )


        # -------------------------------------------------
        # AMRFinderPlus combined mode
        # -------------------------------------------------

        amr_start = time.time()


        amr_result = subprocess.run(
            [
                AMRFINDER_BIN,

                "--protein",
                protein_fasta,

                "--nucleotide",
                nucleotide_fasta,

                "--gff",
                prodigal_gff,

                "--annotation_format",
                "prodigal",

                "--organism",
                AMRFINDER_ORGANISM,

                "--threads",
                "2",

                "--name",
                genome_id,

                "--output",
                combined_output
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=amr_runtime_env
        )


        amrfinder_seconds = (
            time.time()
            - amr_start
        )


        if amr_result.returncode != 0:

            raise RuntimeError(
                "AMRFinderPlus combined mode failed:\n"
                + amr_result.stdout
            )


        # -------------------------------------------------
        # Read combined-mode output
        # -------------------------------------------------

        df_combined = pd.read_csv(
            combined_output,
            sep="\t"
        )


        # -------------------------------------------------
        # Read original nucleotide-only pilot output
        # -------------------------------------------------

        original_output = os.path.join(
            PILOT_CHECKPOINT_DIR,
            f"{genome_id}.amrfinder.tsv"
        )


        df_original = pd.read_csv(
            original_output,
            sep="\t"
        )


        # -------------------------------------------------
        # Compare all reported elements
        # -------------------------------------------------

        original_all = set(
            df_original[
                "Element symbol"
            ]
            .dropna()
            .astype(str)
        )


        combined_all = set(
            df_combined[
                "Element symbol"
            ]
            .dropna()
            .astype(str)
        )


        # -------------------------------------------------
        # Compare AMR-only determinants
        # -------------------------------------------------

        original_amr = set(
            df_original.loc[
                df_original[
                    "Type"
                ] == "AMR",
                "Element symbol"
            ]
            .dropna()
            .astype(str)
        )


        combined_amr = set(
            df_combined.loc[
                df_combined[
                    "Type"
                ] == "AMR",
                "Element symbol"
            ]
            .dropna()
            .astype(str)
        )


        benchmark_records.append(
            {
                "Genome ID":
                    genome_id,

                "Prodigal Seconds":
                    round(
                        prodigal_seconds,
                        2
                    ),

                "AMRFinder Combined Seconds":
                    round(
                        amrfinder_seconds,
                        2
                    ),

                "Total Combined Seconds":
                    round(
                        prodigal_seconds
                        + amrfinder_seconds,
                        2
                    ),

                "Original Total Hits":
                    len(df_original),

                "Combined Total Hits":
                    len(df_combined),

                "Original AMR Features":
                    len(original_amr),

                "Combined AMR Features":
                    len(combined_amr),

                "AMR Features Shared":
                    len(
                        original_amr
                        & combined_amr
                    ),

                "AMR Only Original":
                    len(
                        original_amr
                        - combined_amr
                    ),

                "AMR Only Combined":
                    len(
                        combined_amr
                        - original_amr
                    ),

                "Original-only Symbols":
                    ";".join(
                        sorted(
                            original_amr
                            - combined_amr
                        )
                    ),

                "Combined-only Symbols":
                    ";".join(
                        sorted(
                            combined_amr
                            - original_amr
                        )
                    )
            }
        )


    finally:

        shutil.rmtree(
            temp_dir,
            ignore_errors=True
        )


# =========================================================
# 5. Benchmark table
# =========================================================

df_combined_mode_benchmark = pd.DataFrame(
    benchmark_records
)


df_combined_mode_benchmark.to_csv(
    COMBINED_BENCHMARK_RESULT_PATH,
    index=False
)


print()
print("=" * 70)
print("PRODIGAL + AMRFINDERPLUS COMBINED-MODE BENCHMARK")
print("=" * 70)

display(
    df_combined_mode_benchmark
)


# =========================================================
# 6. Runtime comparison
# =========================================================

mean_prodigal = (
    df_combined_mode_benchmark[
        "Prodigal Seconds"
    ]
    .mean()
)


mean_amrfinder_combined = (
    df_combined_mode_benchmark[
        "AMRFinder Combined Seconds"
    ]
    .mean()
)


mean_total_combined = (
    df_combined_mode_benchmark[
        "Total Combined Seconds"
    ]
    .mean()
)


NUCLEOTIDE_ONLY_MEAN = 35.20


speedup = (
    NUCLEOTIDE_ONLY_MEAN
    / mean_total_combined
)


print()
print(
    f"Mean Prodigal runtime       : "
    f"{mean_prodigal:.2f} s/genome"
)

print(
    f"Mean combined AMRFinder     : "
    f"{mean_amrfinder_combined:.2f} s/genome"
)

print(
    f"Mean total combined runtime : "
    f"{mean_total_combined:.2f} s/genome"
)

print(
    f"Nucleotide-only benchmark   : "
    f"{NUCLEOTIDE_ONLY_MEAN:.2f} s/genome"
)

print(
    f"Runtime speed-up            : "
    f"{speedup:.2f}x"
)


# =========================================================
# 7. Concordance summary
# =========================================================

total_original_only = int(
    df_combined_mode_benchmark[
        "AMR Only Original"
    ]
    .sum()
)


total_combined_only = int(
    df_combined_mode_benchmark[
        "AMR Only Combined"
    ]
    .sum()
)


print()
print(
    "AMR determinant differences across "
    "six pilot genomes:"
)

print(
    f"Original-only occurrences : "
    f"{total_original_only}"
)

print(
    f"Combined-only occurrences : "
    f"{total_combined_only}"
)


print()
print(
    "Benchmark saved to:"
)

print(
    COMBINED_BENCHMARK_RESULT_PATH
)

Installing Prodigal 2.6.3...
Prodigal V2.6.3: February, 2016

PRODIGAL + AMRFINDERPLUS COMBINED-MODE BENCHMARK


,Genome ID,Prodigal Seconds,AMRFinder Combined Seconds,Total Combined Seconds,Original Total Hits,Combined Total Hits,Original AMR Features,Combined AMR Features,AMR Features Shared,AMR Only Original,AMR Only Combined,Original-only Symbols,Combined-only Symbols
0,1284787.3,12.46,97.25,109.72,17,17,16,16,16,0,0,,
1,1284788.3,11.59,98.60,110.19,18,18,17,17,17,0,0,,
2,1284789.4,12.30,96.39,108.69,12,12,12,12,12,0,0,,
3,1284805.4,12.62,99.30,111.92,24,24,22,22,22,0,0,,
4,1284814.3,11.10,94.23,105.32,15,15,14,14,14,0,0,,
5,1284818.4,11.37,96.08,107.46,18,18,18,18,18,0,0,,



Mean Prodigal runtime       : 11.91 s/genome
Mean combined AMRFinder     : 96.98 s/genome
Mean total combined runtime : 108.88 s/genome
Nucleotide-only benchmark   : 35.20 s/genome
Runtime speed-up            : 0.32x

AMR determinant differences across six pilot genomes:
Original-only occurrences : 0
Combined-only occurrences : 0

Benchmark saved to:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/amrfinder_pilot/amrfinder_combined_mode_benchmark.csv


# Cell 19 — Benchmark Prodigal + Protein-Only AMRFinderPlus

In [ ]:
import os
import gzip
import shutil
import subprocess
import tempfile
import time
import pandas as pd


# =========================================================
# 1. Benchmark configuration
# =========================================================

PROTEIN_BENCHMARK_DIR = (
    "/content/amrfinder_protein_benchmark"
)

os.makedirs(
    PROTEIN_BENCHMARK_DIR,
    exist_ok=True
)


PROTEIN_BENCHMARK_RESULT_PATH = os.path.join(
    PILOT_CHECKPOINT_DIR,
    "amrfinder_protein_only_benchmark.csv"
)


# Same six original pilot genomes
df_protein_benchmark = (
    df_amr_pilot[
        [
            "Genome ID",
            "FASTA Path"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(df_protein_benchmark) == 6


# =========================================================
# 2. Run Prodigal + protein-only AMRFinderPlus
# =========================================================

protein_benchmark_records = []


for _, row in df_protein_benchmark.iterrows():

    genome_id = str(
        row["Genome ID"]
    )

    fasta_gz_path = (
        row["FASTA Path"]
    )


    temp_dir = tempfile.mkdtemp(
        prefix=f"{genome_id}_",
        dir=PROTEIN_BENCHMARK_DIR
    )


    nucleotide_fasta = os.path.join(
        temp_dir,
        f"{genome_id}.fna"
    )

    protein_fasta = os.path.join(
        temp_dir,
        f"{genome_id}.faa"
    )

    prodigal_gff = os.path.join(
        temp_dir,
        f"{genome_id}.gff"
    )

    protein_output = os.path.join(
        temp_dir,
        f"{genome_id}.protein_only.tsv"
    )


    try:

        # -------------------------------------------------
        # Decompress assembly
        # -------------------------------------------------

        with gzip.open(
            fasta_gz_path,
            "rb"
        ) as src:

            with open(
                nucleotide_fasta,
                "wb"
            ) as dst:

                shutil.copyfileobj(
                    src,
                    dst
                )


        # -------------------------------------------------
        # Prodigal
        # -------------------------------------------------

        prodigal_start = time.time()


        prodigal_result = subprocess.run(
            [
                PRODIGAL_BIN,

                "-i",
                nucleotide_fasta,

                "-a",
                protein_fasta,

                "-o",
                prodigal_gff,

                "-f",
                "gff",

                "-p",
                "single",

                "-q"
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=amr_runtime_env
        )


        prodigal_seconds = (
            time.time()
            - prodigal_start
        )


        if prodigal_result.returncode != 0:

            raise RuntimeError(
                "Prodigal failed:\n"
                + prodigal_result.stdout
            )


        # -------------------------------------------------
        # Protein-only AMRFinderPlus
        # -------------------------------------------------

        amr_start = time.time()


        amr_result = subprocess.run(
            [
                AMRFINDER_BIN,

                "--protein",
                protein_fasta,

                "--organism",
                AMRFINDER_ORGANISM,

                "--threads",
                "2",

                "--name",
                genome_id,

                "--output",
                protein_output
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=amr_runtime_env
        )


        protein_amr_seconds = (
            time.time()
            - amr_start
        )


        if amr_result.returncode != 0:

            raise RuntimeError(
                "Protein-only AMRFinderPlus failed:\n"
                + amr_result.stdout
            )


        # -------------------------------------------------
        # Read protein-only output
        # -------------------------------------------------

        df_protein = pd.read_csv(
            protein_output,
            sep="\t"
        )


        # -------------------------------------------------
        # Original nucleotide-only pilot
        # -------------------------------------------------

        original_output = os.path.join(
            PILOT_CHECKPOINT_DIR,
            f"{genome_id}.amrfinder.tsv"
        )


        df_original = pd.read_csv(
            original_output,
            sep="\t"
        )


        # -------------------------------------------------
        # AMR-only determinant sets
        # -------------------------------------------------

        original_amr = set(
            df_original.loc[
                df_original[
                    "Type"
                ] == "AMR",
                "Element symbol"
            ]
            .dropna()
            .astype(str)
        )


        protein_amr = set(
            df_protein.loc[
                df_protein[
                    "Type"
                ] == "AMR",
                "Element symbol"
            ]
            .dropna()
            .astype(str)
        )


        original_only = (
            original_amr
            - protein_amr
        )

        protein_only = (
            protein_amr
            - original_amr
        )


        # -------------------------------------------------
        # Mutation-specific comparison
        # -------------------------------------------------

        mutation_subtypes = {
            "POINT",
            "POINT_DISRUPT"
        }


        original_mutations = set(
            df_original.loc[
                (
                    df_original[
                        "Type"
                    ] == "AMR"
                )
                &
                (
                    df_original[
                        "Subtype"
                    ]
                    .isin(
                        mutation_subtypes
                    )
                ),
                "Element symbol"
            ]
            .dropna()
            .astype(str)
        )


        protein_mutations = set(
            df_protein.loc[
                (
                    df_protein[
                        "Type"
                    ] == "AMR"
                )
                &
                (
                    df_protein[
                        "Subtype"
                    ]
                    .isin(
                        mutation_subtypes
                    )
                ),
                "Element symbol"
            ]
            .dropna()
            .astype(str)
        )


        protein_benchmark_records.append(
            {
                "Genome ID":
                    genome_id,

                "Prodigal Seconds":
                    round(
                        prodigal_seconds,
                        2
                    ),

                "Protein AMRFinder Seconds":
                    round(
                        protein_amr_seconds,
                        2
                    ),

                "Total Protein Workflow Seconds":
                    round(
                        prodigal_seconds
                        + protein_amr_seconds,
                        2
                    ),

                "Original AMR Features":
                    len(
                        original_amr
                    ),

                "Protein AMR Features":
                    len(
                        protein_amr
                    ),

                "AMR Features Shared":
                    len(
                        original_amr
                        & protein_amr
                    ),

                "AMR Only Original":
                    len(
                        original_only
                    ),

                "AMR Only Protein":
                    len(
                        protein_only
                    ),

                "Original Mutation Features":
                    len(
                        original_mutations
                    ),

                "Protein Mutation Features":
                    len(
                        protein_mutations
                    ),

                "Mutation Features Shared":
                    len(
                        original_mutations
                        & protein_mutations
                    ),

                "Original-only Symbols":
                    ";".join(
                        sorted(
                            original_only
                        )
                    ),

                "Protein-only Symbols":
                    ";".join(
                        sorted(
                            protein_only
                        )
                    )
            }
        )


    finally:

        shutil.rmtree(
            temp_dir,
            ignore_errors=True
        )


# =========================================================
# 3. Benchmark results
# =========================================================

df_protein_mode_benchmark = pd.DataFrame(
    protein_benchmark_records
)


df_protein_mode_benchmark.to_csv(
    PROTEIN_BENCHMARK_RESULT_PATH,
    index=False
)


print()
print("=" * 70)
print("PRODIGAL + PROTEIN-ONLY AMRFINDERPLUS BENCHMARK")
print("=" * 70)

display(
    df_protein_mode_benchmark
)


# =========================================================
# 4. Runtime summary
# =========================================================

mean_prodigal = (
    df_protein_mode_benchmark[
        "Prodigal Seconds"
    ]
    .mean()
)


mean_protein_amr = (
    df_protein_mode_benchmark[
        "Protein AMRFinder Seconds"
    ]
    .mean()
)


mean_protein_total = (
    df_protein_mode_benchmark[
        "Total Protein Workflow Seconds"
    ]
    .mean()
)


NUCLEOTIDE_ONLY_MEAN = 35.20


speedup = (
    NUCLEOTIDE_ONLY_MEAN
    / mean_protein_total
)


# =========================================================
# 5. Concordance summary
# =========================================================

original_only_total = int(
    df_protein_mode_benchmark[
        "AMR Only Original"
    ]
    .sum()
)


protein_only_total = int(
    df_protein_mode_benchmark[
        "AMR Only Protein"
    ]
    .sum()
)


original_mutations_total = int(
    df_protein_mode_benchmark[
        "Original Mutation Features"
    ]
    .sum()
)


shared_mutations_total = int(
    df_protein_mode_benchmark[
        "Mutation Features Shared"
    ]
    .sum()
)


print()
print(
    f"Mean Prodigal runtime          : "
    f"{mean_prodigal:.2f} s/genome"
)

print(
    f"Mean protein AMRFinder runtime : "
    f"{mean_protein_amr:.2f} s/genome"
)

print(
    f"Mean total protein workflow    : "
    f"{mean_protein_total:.2f} s/genome"
)

print(
    f"Nucleotide-only benchmark      : "
    f"{NUCLEOTIDE_ONLY_MEAN:.2f} s/genome"
)

print(
    f"Runtime speed-up               : "
    f"{speedup:.2f}x"
)


print()
print(
    "AMR determinant differences:"
)

print(
    f"Original-only occurrences      : "
    f"{original_only_total}"
)

print(
    f"Protein-only occurrences       : "
    f"{protein_only_total}"
)


print()
print(
    "Mutation concordance:"
)

print(
    f"Original mutation occurrences  : "
    f"{original_mutations_total}"
)

print(
    f"Shared mutation occurrences    : "
    f"{shared_mutations_total}"
)


if original_mutations_total > 0:

    mutation_recall = (
        shared_mutations_total
        / original_mutations_total
    )

    print(
        f"Mutation recall vs nucleotide  : "
        f"{mutation_recall:.3f}"
    )


print()
print(
    "Benchmark saved to:"
)

print(
    PROTEIN_BENCHMARK_RESULT_PATH
)


PRODIGAL + PROTEIN-ONLY AMRFINDERPLUS BENCHMARK


,Genome ID,Prodigal Seconds,Protein AMRFinder Seconds,Total Protein Workflow Seconds,Original AMR Features,Protein AMR Features,AMR Features Shared,AMR Only Original,AMR Only Protein,Original Mutation Features,Protein Mutation Features,Mutation Features Shared,Original-only Symbols,Protein-only Symbols
0,1284787.3,14.70,62.86,77.56,16,15,15,1,0,4,3,3,ompK35_E42RfsTer47,
1,1284788.3,11.54,62.43,73.98,17,15,15,2,0,4,3,3,blaOXA;ompK35_E42RfsTer47,
2,1284789.4,11.81,61.28,73.08,12,10,10,2,0,5,4,4,blaOXA;ompK35_E42RfsTer47,
3,1284805.4,12.50,62.98,75.49,22,21,20,2,1,4,3,3,estX-3;ompK35_E42RfsTer47,estX
4,1284814.3,12.26,60.60,72.86,14,13,13,1,0,4,3,3,ompK35_E42RfsTer47,
5,1284818.4,12.36,62.32,74.68,18,16,16,2,0,5,4,4,dfrA14;mgrB_C28Y,



Mean Prodigal runtime          : 12.53 s/genome
Mean protein AMRFinder runtime : 62.08 s/genome
Mean total protein workflow    : 74.61 s/genome
Nucleotide-only benchmark      : 35.20 s/genome
Runtime speed-up               : 0.47x

AMR determinant differences:
Original-only occurrences      : 10
Protein-only occurrences       : 1

Mutation concordance:
Original mutation occurrences  : 26
Shared mutation occurrences    : 20
Mutation recall vs nucleotide  : 0.769

Benchmark saved to:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/amrfinder_pilot/amrfinder_protein_only_benchmark.csv


# Cell 20 — Finalize AMRFinderPlus Screening Strategy

In [ ]:
import os
import pandas as pd


# =========================================================
# 1. Persistent benchmark paths
# =========================================================

NUCLEOTIDE_BENCHMARK_MEAN = 35.20


COMBINED_BENCHMARK_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_pilot",
    "amrfinder_combined_mode_benchmark.csv"
)


PROTEIN_BENCHMARK_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_pilot",
    "amrfinder_protein_only_benchmark.csv"
)


STRATEGY_DECISION_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "amrfinder_screening_strategy_decision.csv"
)


# =========================================================
# 2. Reload benchmark outputs from Drive
# =========================================================

assert os.path.exists(
    COMBINED_BENCHMARK_PATH
), "Combined-mode benchmark file is missing."


assert os.path.exists(
    PROTEIN_BENCHMARK_PATH
), "Protein-only benchmark file is missing."


df_combined_benchmark_final = pd.read_csv(
    COMBINED_BENCHMARK_PATH,
    dtype={
        "Genome ID": str
    }
)


df_protein_benchmark_final = pd.read_csv(
    PROTEIN_BENCHMARK_PATH,
    dtype={
        "Genome ID": str
    }
)


# =========================================================
# 3. Compute final benchmark statistics
# =========================================================

combined_mean = (
    df_combined_benchmark_final[
        "Total Combined Seconds"
    ]
    .mean()
)


protein_mean = (
    df_protein_benchmark_final[
        "Total Protein Workflow Seconds"
    ]
    .mean()
)


combined_original_only = int(
    df_combined_benchmark_final[
        "AMR Only Original"
    ]
    .sum()
)


combined_new_only = int(
    df_combined_benchmark_final[
        "AMR Only Combined"
    ]
    .sum()
)


protein_original_only = int(
    df_protein_benchmark_final[
        "AMR Only Original"
    ]
    .sum()
)


protein_new_only = int(
    df_protein_benchmark_final[
        "AMR Only Protein"
    ]
    .sum()
)


original_mutations = int(
    df_protein_benchmark_final[
        "Original Mutation Features"
    ]
    .sum()
)


shared_mutations = int(
    df_protein_benchmark_final[
        "Mutation Features Shared"
    ]
    .sum()
)


mutation_recall = (
    shared_mutations
    / original_mutations
)


# =========================================================
# 4. Estimate full-cohort nucleotide runtime
# =========================================================

TOTAL_GENOMES = 4233


estimated_serial_hours = (
    TOTAL_GENOMES
    * NUCLEOTIDE_BENCHMARK_MEAN
    / 3600
)


# =========================================================
# 5. Record screening-strategy decision
# =========================================================

df_strategy_decision = pd.DataFrame(
    [
        {
            "Reference Method":
                "AMRFinderPlus nucleotide-only",

            "AMRFinderPlus Version":
                "4.2.7",

            "Database Version":
                "2026-08-07.1",

            "Organism":
                "Klebsiella_pneumoniae",

            "Nucleotide Runtime Seconds per Genome":
                NUCLEOTIDE_BENCHMARK_MEAN,

            "Combined Runtime Seconds per Genome":
                round(
                    combined_mean,
                    2
                ),

            "Protein Runtime Seconds per Genome":
                round(
                    protein_mean,
                    2
                ),

            "Combined Original-only AMR Occurrences":
                combined_original_only,

            "Combined-only AMR Occurrences":
                combined_new_only,

            "Protein Original-only AMR Occurrences":
                protein_original_only,

            "Protein-only AMR Occurrences":
                protein_new_only,

            "Protein Mutation Recall":
                round(
                    mutation_recall,
                    3
                ),

            "Estimated Serial Full-Cohort Hours":
                round(
                    estimated_serial_hours,
                    1
                ),

            "Full Local Screening Status":
                "Deferred",

            "Next Strategy":
                (
                    "Evaluate BV-BRC precomputed curated "
                    "AMR annotations before full local screening"
                )
        }
    ]
)


df_strategy_decision.to_csv(
    STRATEGY_DECISION_PATH,
    index=False
)


# =========================================================
# 6. Final notebook summary
# =========================================================

print("=" * 70)
print("AMRFINDERPLUS SCREENING STRATEGY — FINAL DECISION")
print("=" * 70)

print(
    f"Nucleotide-only runtime      : "
    f"{NUCLEOTIDE_BENCHMARK_MEAN:.2f} s/genome"
)

print(
    f"Combined-mode runtime        : "
    f"{combined_mean:.2f} s/genome"
)

print(
    f"Protein-only runtime         : "
    f"{protein_mean:.2f} s/genome"
)

print()

print(
    f"Combined-mode lost AMR hits  : "
    f"{combined_original_only}"
)

print(
    f"Protein-only lost AMR hits   : "
    f"{protein_original_only}"
)

print(
    f"Protein mutation recall      : "
    f"{mutation_recall:.3f}"
)

print()

print(
    f"Estimated nucleotide-only "
    f"serial runtime for 4,233 genomes: "
    f"{estimated_serial_hours:.1f} h"
)

print()

print(
    "REFERENCE METHOD:"
)

print(
    "AMRFinderPlus nucleotide-only"
)

print()

print(
    "FULL LOCAL SCREENING:"
)

print(
    "DEFERRED — evaluate precomputed BV-BRC "
    "AMR annotations first."
)

print()

print(
    "Decision record saved to:"
)

print(
    STRATEGY_DECISION_PATH
)

print()

print(
    "PASS: File 03 genomic retrieval and "
    "AMR screening strategy are finalized."
)

AMRFINDERPLUS SCREENING STRATEGY — FINAL DECISION
Nucleotide-only runtime      : 35.20 s/genome
Combined-mode runtime        : 108.88 s/genome
Protein-only runtime         : 74.61 s/genome

Combined-mode lost AMR hits  : 0
Protein-only lost AMR hits   : 10
Protein mutation recall      : 0.769

Estimated nucleotide-only serial runtime for 4,233 genomes: 41.4 h

REFERENCE METHOD:
AMRFinderPlus nucleotide-only

FULL LOCAL SCREENING:
DEFERRED — evaluate precomputed BV-BRC AMR annotations first.

Decision record saved to:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/amrfinder_screening_strategy_decision.csv

PASS: File 03 genomic retrieval and AMR screening strategy are finalized.


# Cell 7 — Retrieve Precomputed NCBI MicroBIGG-E AMRFinderPlus Results

In [1]:
import os
import time
import requests
import pandas as pd


# =========================================================
# 1. NCBI MicroBIGG-E query endpoint
# =========================================================

MICROBIGGE_URL = (
    "https://www.ncbi.nlm.nih.gov/pathogens/microbigge/"
)

ncbi_session = requests.Session()

ncbi_session.headers.update(
    {
        "User-Agent": (
            "AMR-Genome-ML/1.0 "
            "(academic research)"
        )
    }
)


# =========================================================
# 2. Pilot assembly accessions
# =========================================================

pilot_assemblies = (
    df_pilot_ncbi_mapping[
        [
            "Genome ID",
            "Assembly Accession"
        ]
    ]
    .dropna()
    .copy()
)


assert len(pilot_assemblies) == 6


# =========================================================
# 3. Query MicroBIGG-E CSV export for each assembly
# =========================================================

microbigge_records = []

microbigge_audit = []


for i, row in pilot_assemblies.iterrows():

    genome_id = str(
        row["Genome ID"]
    )

    assembly = str(
        row["Assembly Accession"]
    )


    # NCBI table export query
    params = {
        "q": f"asm_acc:{assembly}",
        "format": "csv"
    }


    start_time = time.time()


    try:

        response = ncbi_session.get(
            MICROBIGGE_URL,
            params=params,
            timeout=90
        )

        response.raise_for_status()


        content_type = (
            response.headers.get(
                "Content-Type",
                ""
            )
        )


        elapsed = (
            time.time()
            - start_time
        )


        # -------------------------------------------------
        # Save raw response for inspection
        # -------------------------------------------------

        raw_path = os.path.join(
            KNOWN_AMR_DIR,
            f"microbigge_{assembly}.csv"
        )


        with open(
            raw_path,
            "wb"
        ) as f:

            f.write(
                response.content
            )


        # -------------------------------------------------
        # Try reading as CSV
        # -------------------------------------------------

        try:

            df_result = pd.read_csv(
                raw_path
            )

        except Exception:

            df_result = pd.DataFrame()


        # -------------------------------------------------
        # Attach linkage columns
        # -------------------------------------------------

        if not df_result.empty:

            df_result[
                "Genome ID"
            ] = genome_id

            df_result[
                "Assembly Accession"
            ] = assembly


            microbigge_records.append(
                df_result
            )


        microbigge_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Assembly Accession":
                    assembly,

                "Status":
                    "PASS",

                "Rows Retrieved":
                    len(
                        df_result
                    ),

                "HTTP Content Type":
                    content_type,

                "Runtime Seconds":
                    round(
                        elapsed,
                        2
                    ),

                "Raw File":
                    raw_path,

                "Error":
                    None
            }
        )


        print(
            f"[{i + 1}/6] "
            f"{assembly} | "
            f"{len(df_result):,} rows | "
            f"{elapsed:.2f} s"
        )


    except Exception as exc:

        microbigge_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Assembly Accession":
                    assembly,

                "Status":
                    "FAIL",

                "Rows Retrieved":
                    0,

                "HTTP Content Type":
                    None,

                "Runtime Seconds":
                    None,

                "Raw File":
                    None,

                "Error":
                    str(exc)
            }
        )


        print(
            f"[{i + 1}/6] "
            f"{assembly} | FAIL"
        )


# =========================================================
# 4. Combine results
# =========================================================

if microbigge_records:

    df_microbigge_pilot = pd.concat(
        microbigge_records,
        ignore_index=True
    )

else:

    df_microbigge_pilot = pd.DataFrame()


df_microbigge_audit = pd.DataFrame(
    microbigge_audit
)


# =========================================================
# 5. Summary
# =========================================================

print()
print("=" * 70)
print("NCBI MICROBIGG-E PILOT RETRIEVAL")
print("=" * 70)

print(
    f"Pilot assemblies requested : "
    f"{len(pilot_assemblies)}"
)

print(
    f"Successful HTTP requests   : "
    f"{df_microbigge_audit['Status'].eq('PASS').sum()}/6"
)

print(
    f"Total rows retrieved       : "
    f"{len(df_microbigge_pilot):,}"
)


if not df_microbigge_pilot.empty:

    print()

    print(
        "Returned columns:"
    )

    print(
        df_microbigge_pilot
        .columns
        .tolist()
    )


# =========================================================
# 6. Audit table
# =========================================================

print()
display(
    df_microbigge_audit
)


# =========================================================
# 7. Preview results
# =========================================================

if not df_microbigge_pilot.empty:

    print()
    print(
        "Example MicroBIGG-E records:"
    )

    display(
        df_microbigge_pilot
        .head(30)
    )


# =========================================================
# 8. Save combined pilot results
# =========================================================

MICROBIGGE_PILOT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_microbigge_pilot_raw.csv"
)

MICROBIGGE_AUDIT_PATH = os.path.join(
    KNOWN_AMR_DIR,
    "ncbi_microbigge_pilot_audit.csv"
)


df_microbigge_pilot.to_csv(
    MICROBIGGE_PILOT_PATH,
    index=False
)


df_microbigge_audit.to_csv(
    MICROBIGGE_AUDIT_PATH,
    index=False
)


print()
print(
    "Pilot MicroBIGG-E data saved to:"
)

print(
    MICROBIGGE_PILOT_PATH
)

NameError: name 'df_pilot_ncbi_mapping' is not defined